In [ ]:
# Image:       224 × 224
# Patch:       16 × 16
# Tokens:      196
# Dimension:   384
# Layers:      6
# Heads:       6
# Anchors:     16
# Dataset:     CIFAR-100

# CIFAR-100
#    ↓
# Patch Embedding
#    ↓
# Local Attention ──────┐
#                      ├→ Fusion → FFN
# Patch → Anchors → Global
#    ↑                  │
#    └── Residual ──────┘
#    ↓
# Classifier

In [1]:
import torch
import torchvision
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [4]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Seed:", SEED)

Device: cuda
Seed: 42


In [5]:
IMG_SIZE = 224
PATCH_SIZE = 16
BATCH_SIZE = 128
NUM_CLASSES = 100

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

train_dataset = torchvision.datasets.CIFAR100(
    root="/kaggle/working/data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR100(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=test_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Number of classes:", NUM_CLASSES)
print("Image size:", IMG_SIZE)
print("Patch size:", PATCH_SIZE)
print("Number of patches:", (IMG_SIZE // PATCH_SIZE) ** 2)

100%|██████████| 169M/169M [25:57<00:00, 109kB/s]  


Train samples: 50000
Test samples: 10000
Number of classes: 100
Image size: 224
Patch size: 16
Number of patches: 196


In [6]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Image range:", images.min().item(), "to", images.max().item())

Images shape: torch.Size([128, 3, 224, 224])
Labels shape: torch.Size([128])
Image dtype: torch.float32
Label dtype: torch.int64
Image range: -1.8974658250808716 to 2.025352954864502


In [7]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=384):
        super().__init__()

        assert image_size % patch_size == 0

        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2

        self.projection = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.projection(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


patch_embed = PatchEmbedding(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    embed_dim=384
).to(DEVICE)

images_gpu = images.to(DEVICE)

patch_tokens = patch_embed(images_gpu)

print("Input shape:", images_gpu.shape)
print("Patch token shape:", patch_tokens.shape)
print("Number of patches:", patch_embed.num_patches)
print("Embedding dimension:", patch_tokens.shape[-1])

Input shape: torch.Size([128, 3, 224, 224])
Patch token shape: torch.Size([128, 196, 384])
Number of patches: 196
Embedding dimension: 384


In [8]:
class ViTInputEmbedding(nn.Module):
    def __init__(self, num_patches, embed_dim=384):
        super().__init__()

        self.cls_token = nn.Parameter(
            torch.randn(1, 1, embed_dim) * 0.02
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(1, num_patches + 1, embed_dim) * 0.02
        )

    def forward(self, x):
        batch_size = x.size(0)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat([cls_tokens, x], dim=1)

        x = x + self.pos_embedding

        return x


input_embedding = ViTInputEmbedding(
    num_patches=patch_embed.num_patches,
    embed_dim=384
).to(DEVICE)

tokens = input_embedding(patch_tokens)

print("Patch tokens:", patch_tokens.shape)
print("Tokens after CLS:", tokens.shape)
print("CLS token shape:", tokens[:, :1].shape)
print("Patch token shape:", tokens[:, 1:].shape)
print("Positional embedding:", input_embedding.pos_embedding.shape)

Patch tokens: torch.Size([128, 196, 384])
Tokens after CLS: torch.Size([128, 197, 384])
CLS token shape: torch.Size([128, 1, 384])
Patch token shape: torch.Size([128, 196, 384])
Positional embedding: torch.Size([1, 197, 384])


In [9]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=384, num_heads=6, dropout=0.0):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.projection = nn.Linear(embed_dim, embed_dim)

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        qkv = self.qkv(x)

        qkv = qkv.reshape(
            batch_size,
            num_tokens,
            3,
            self.num_heads,
            self.head_dim
        )

        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv.unbind(0)

        attention_scores = (q @ k.transpose(-2, -1)) * self.scale

        attention_weights = attention_scores.softmax(dim=-1)

        attention_weights = self.attention_dropout(attention_weights)

        x = attention_weights @ v

        x = x.transpose(1, 2).reshape(
            batch_size,
            num_tokens,
            embed_dim
        )

        x = self.projection(x)

        x = self.projection_dropout(x)

        return x


attention = MultiHeadSelfAttention(
    embed_dim=384,
    num_heads=6
).to(DEVICE)

attention_output = attention(tokens)

print("Input:", tokens.shape)
print("Output:", attention_output.shape)
print("Attention parameters:", sum(p.numel() for p in attention.parameters()))

Input: torch.Size([128, 197, 384])
Output: torch.Size([128, 197, 384])
Attention parameters: 591360


In [10]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim=384, hidden_dim=1536, dropout=0.0):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)


ffn = FeedForward(
    embed_dim=384,
    hidden_dim=1536
).to(DEVICE)

ffn_output = ffn(tokens)

print("Input:", tokens.shape)
print("Output:", ffn_output.shape)
print("FFN parameters:", sum(p.numel() for p in ffn.parameters()))

Input: torch.Size([128, 197, 384])
Output: torch.Size([128, 197, 384])
FFN parameters: 1181568


In [11]:
class TransformerBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        num_heads=6,
        mlp_ratio=4.0,
        dropout=0.0
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attention = MultiHeadSelfAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        hidden_dim = int(embed_dim * mlp_ratio)

        self.ffn = FeedForward(
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            dropout=dropout
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


transformer_block = TransformerBlock(
    embed_dim=384,
    num_heads=6,
    mlp_ratio=4.0
).to(DEVICE)

block_output = transformer_block(tokens)

print("Input:", tokens.shape)
print("Output:", block_output.shape)
print(
    "Block parameters:",
    sum(p.numel() for p in transformer_block.parameters())
)

Input: torch.Size([128, 197, 384])
Output: torch.Size([128, 197, 384])
Block parameters: 1774464


In [12]:
class ViT(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        depth=6,
        num_heads=6,
        mlp_ratio=4.0
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            num_patches=self.patch_embed.num_patches,
            embed_dim=embed_dim
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)

        x = self.input_embedding(x)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        cls_token = x[:, 0]

        return self.head(cls_token)


vit = ViT(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_classes=NUM_CLASSES,
    embed_dim=384,
    depth=6,
    num_heads=6
).to(DEVICE)

total_params = sum(
    p.numel() for p in vit.parameters()
)

trainable_params = sum(
    p.numel()
    for p in vit.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

with torch.no_grad():
    logits = vit(images_gpu)

print("Input:", images_gpu.shape)
print("Logits:", logits.shape)

Total parameters: 11057380
Trainable parameters: 11057380
Input: torch.Size([128, 3, 224, 224])
Logits: torch.Size([128, 100])


In [13]:
import time

def benchmark_model(model, input_tensor, warmup=10, runs=50):
    model.eval()

    for _ in range(warmup):
        with torch.no_grad():
            _ = model(input_tensor)

    torch.cuda.synchronize()

    start_memory = torch.cuda.memory_allocated()

    start = time.perf_counter()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    peak_memory = torch.cuda.max_memory_allocated()

    avg_time = elapsed / runs

    return avg_time, peak_memory


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

avg_time, peak_memory = benchmark_model(
    vit,
    images_gpu
)

print("Parameters:", sum(p.numel() for p in vit.parameters()))
print("Average inference time:", round(avg_time * 1000, 3), "ms")
print("Peak GPU memory:", round(peak_memory / 1024**2, 2), "MB")
print("Throughput:", round(128 / avg_time, 2), "images/sec")

Parameters: 11057380
Average inference time: 201.345 ms
Peak GPU memory: 2039.25 MB
Throughput: 635.73 images/sec


In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    vit.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / total
    accuracy = 100.0 * correct / total

    return average_loss, accuracy


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)

        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / total
    accuracy = 100.0 * correct / total

    return average_loss, accuracy


print("Training utilities ready.")
print("Initial learning rate:", optimizer.param_groups[0]["lr"])

Training utilities ready.
Initial learning rate: 0.0003


In [15]:
import time

start_time = time.perf_counter()

train_loss, train_acc = train_one_epoch(
    vit,
    train_loader,
    optimizer,
    criterion,
    DEVICE
)

test_loss, test_acc = evaluate(
    vit,
    test_loader,
    criterion,
    DEVICE
)

scheduler.step()

elapsed = time.perf_counter() - start_time

print("Epoch: 1")
print("Train Loss:", round(train_loss, 4))
print("Train Accuracy:", round(train_acc, 2), "%")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_acc, 2), "%")
print("Epoch Time:", round(elapsed / 60, 2), "minutes")
print("Learning Rate:", optimizer.param_groups[0]["lr"])

Epoch: 1
Train Loss: 3.7989
Train Accuracy: 11.67 %
Test Loss: 3.4435
Test Accuracy: 17.56 %
Epoch Time: 4.5 minutes
Learning Rate: 0.00029265847744427303


In [16]:
scaler = torch.amp.GradScaler("cuda")

def train_one_epoch_amp(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate_amp(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total


print("AMP training enabled.")

AMP training enabled.


In [17]:
import time

torch.cuda.empty_cache()

start_time = time.perf_counter()

train_loss_amp, train_acc_amp = train_one_epoch_amp(
    vit,
    train_loader,
    optimizer,
    criterion,
    DEVICE
)

torch.cuda.synchronize()

elapsed = time.perf_counter() - start_time

print("AMP Train Loss:", round(train_loss_amp, 4))
print("AMP Train Accuracy:", round(train_acc_amp, 2), "%")
print("AMP Epoch Time:", round(elapsed / 60, 2), "minutes")
print("Speedup vs FP32:", round(4.5 / (elapsed / 60), 2), "x")

AMP Train Loss: 3.2398
AMP Train Accuracy: 20.85 %
AMP Epoch Time: 1.69 minutes
Speedup vs FP32: 2.66 x


In [18]:
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

vit = ViT(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_classes=NUM_CLASSES,
    embed_dim=384,
    depth=6,
    num_heads=6
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    vit.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

scaler = torch.amp.GradScaler("cuda")

print("Baseline reset.")
print("Parameters:", sum(p.numel() for p in vit.parameters()))
print("Learning rate:", optimizer.param_groups[0]["lr"])

Baseline reset.
Parameters: 11057380
Learning rate: 0.0003


In [19]:
import time

history = []

for epoch in range(1, 11):
    start_time = time.perf_counter()

    train_loss, train_acc = train_one_epoch_amp(
        vit,
        train_loader,
        optimizer,
        criterion,
        DEVICE
    )

    test_loss, test_acc = evaluate_amp(
        vit,
        test_loader,
        criterion,
        DEVICE
    )

    scheduler.step()

    elapsed = time.perf_counter() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "lr": optimizer.param_groups[0]["lr"],
        "time": elapsed / 60
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}% | "
        f"Time: {elapsed / 60:.2f} min"
    )

Epoch 01 | Train Loss: 3.8069 | Train Acc: 11.39% | Test Loss: 3.4749 | Test Acc: 16.53% | Time: 1.88 min
Epoch 02 | Train Loss: 3.2479 | Train Acc: 20.39% | Test Loss: 3.1069 | Test Acc: 23.41% | Time: 1.87 min
Epoch 03 | Train Loss: 2.9299 | Train Acc: 26.66% | Test Loss: 2.8685 | Test Acc: 28.23% | Time: 1.89 min
Epoch 04 | Train Loss: 2.6607 | Train Acc: 31.88% | Test Loss: 2.6480 | Test Acc: 32.94% | Time: 1.89 min
Epoch 05 | Train Loss: 2.3854 | Train Acc: 37.42% | Test Loss: 2.4874 | Test Acc: 36.21% | Time: 1.87 min
Epoch 06 | Train Loss: 2.1180 | Train Acc: 43.63% | Test Loss: 2.3656 | Test Acc: 39.31% | Time: 1.87 min
Epoch 07 | Train Loss: 1.8403 | Train Acc: 50.15% | Test Loss: 2.2406 | Test Acc: 42.22% | Time: 1.87 min
Epoch 08 | Train Loss: 1.5713 | Train Acc: 57.09% | Test Loss: 2.1674 | Test Acc: 43.75% | Time: 1.89 min
Epoch 09 | Train Loss: 1.3464 | Train Acc: 63.39% | Test Loss: 2.1268 | Test Acc: 45.14% | Time: 1.89 min
Epoch 10 | Train Loss: 1.2124 | Train Acc: 67.

# Window Partitioning

In [20]:
def window_partition(x, window_size):
    batch_size, height, width, channels = x.shape

    x = x.view(
        batch_size,
        height // window_size,
        window_size,
        width // window_size,
        window_size,
        channels
    )

    windows = x.permute(
        0, 1, 3, 2, 4, 5
    ).contiguous()

    windows = windows.view(
        -1,
        window_size,
        window_size,
        channels
    )

    return windows


def window_reverse(windows, window_size, height, width):
    num_windows = (height // window_size) * (width // window_size)

    batch_size = windows.shape[0] // num_windows
    channels = windows.shape[-1]

    x = windows.view(
        batch_size,
        height // window_size,
        width // window_size,
        window_size,
        window_size,
        channels
    )

    x = x.permute(
        0, 1, 3, 2, 4, 5
    ).contiguous()

    x = x.view(
        batch_size,
        height,
        width,
        channels
    )

    return x


patch_grid = patch_tokens.view(
    patch_tokens.shape[0],
    14,
    14,
    384
)

windows = window_partition(
    patch_grid,
    window_size=7
)

reconstructed = window_reverse(
    windows,
    window_size=7,
    height=14,
    width=14
)

print("Patch grid:", patch_grid.shape)
print("Windows:", windows.shape)
print("Reconstructed:", reconstructed.shape)
print(
    "Reconstruction error:",
    (patch_grid - reconstructed).abs().max().item()
)

Patch grid: torch.Size([128, 14, 14, 384])
Windows: torch.Size([512, 7, 7, 384])
Reconstructed: torch.Size([128, 14, 14, 384])
Reconstruction error: 0.0


# Local Window Attention

In [21]:
class LocalWindowAttention(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        num_heads=6,
        window_size=7,
        dropout=0.0
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.window_size = window_size
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(
            embed_dim,
            embed_dim * 3
        )

        self.projection = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, height, width, embed_dim = x.shape

        windows = window_partition(
            x,
            self.window_size
        )

        windows = windows.view(
            -1,
            self.window_size * self.window_size,
            embed_dim
        )

        qkv = self.qkv(windows)

        qkv = qkv.reshape(
            qkv.shape[0],
            qkv.shape[1],
            3,
            self.num_heads,
            self.head_dim
        )

        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv.unbind(0)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention_weights = attention_scores.softmax(
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(
            1, 2
        ).reshape(
            windows.shape[0],
            windows.shape[1],
            embed_dim
        )

        output = self.projection(output)
        output = self.projection_dropout(output)

        output = output.view(
            -1,
            self.window_size,
            self.window_size,
            embed_dim
        )

        output = window_reverse(
            output,
            self.window_size,
            height,
            width
        )

        return output


local_attention = LocalWindowAttention(
    embed_dim=384,
    num_heads=6,
    window_size=7
).to(DEVICE)

patch_grid_gpu = patch_tokens.view(
    patch_tokens.shape[0],
    14,
    14,
    384
)

with torch.no_grad():
    local_output = local_attention(patch_grid_gpu)

print("Input:", patch_grid_gpu.shape)
print("Output:", local_output.shape)
print(
    "Local attention parameters:",
    sum(p.numel() for p in local_attention.parameters())
)

Input: torch.Size([128, 14, 14, 384])
Output: torch.Size([128, 14, 14, 384])
Local attention parameters: 591360


# Anchor Cross Attention

In [22]:
class PatchToAnchorAttention(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        num_heads=6,
        num_anchors=16,
        dropout=0.0
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_anchors = num_anchors
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.anchor_tokens = nn.Parameter(
            torch.randn(1, num_anchors, embed_dim) * 0.02
        )

        self.q_projection = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.kv_projection = nn.Linear(
            embed_dim,
            embed_dim * 2
        )

        self.output_projection = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size = x.shape[0]

        anchors = self.anchor_tokens.expand(
            batch_size,
            -1,
            -1
        )

        q = self.q_projection(anchors)

        kv = self.kv_projection(x)

        k, v = kv.chunk(2, dim=-1)

        q = q.view(
            batch_size,
            self.num_anchors,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            x.shape[1],
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            x.shape[1],
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention_weights = attention_scores.softmax(
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(
            1,
            2
        ).reshape(
            batch_size,
            self.num_anchors,
            self.embed_dim
        )

        output = self.output_projection(output)
        output = self.projection_dropout(output)

        return output


patch_to_anchor = PatchToAnchorAttention(
    embed_dim=384,
    num_heads=6,
    num_anchors=16
).to(DEVICE)

patch_tokens_flat = patch_tokens

with torch.no_grad():
    anchors = patch_to_anchor(patch_tokens_flat)

print("Patch tokens:", patch_tokens_flat.shape)
print("Anchors:", anchors.shape)
print(
    "Parameters:",
    sum(p.numel() for p in patch_to_anchor.parameters())
)

Patch tokens: torch.Size([128, 196, 384])
Anchors: torch.Size([128, 16, 384])
Parameters: 597504


# Anchor Self Attention

In [23]:
class AnchorSelfAttention(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        num_heads=6,
        dropout=0.0
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(
            embed_dim,
            embed_dim * 3
        )

        self.projection = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, num_anchors, embed_dim = x.shape

        qkv = self.qkv(x)

        qkv = qkv.reshape(
            batch_size,
            num_anchors,
            3,
            self.num_heads,
            self.head_dim
        )

        qkv = qkv.permute(
            2,
            0,
            3,
            1,
            4
        )

        q, k, v = qkv.unbind(0)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention_weights = attention_scores.softmax(
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(
            1,
            2
        ).reshape(
            batch_size,
            num_anchors,
            embed_dim
        )

        output = self.projection(output)
        output = self.projection_dropout(output)

        return output


anchor_attention = AnchorSelfAttention(
    embed_dim=384,
    num_heads=6
).to(DEVICE)

with torch.no_grad():
    global_anchors = anchor_attention(anchors)

print("Input anchors:", anchors.shape)
print("Output anchors:", global_anchors.shape)
print(
    "Parameters:",
    sum(p.numel() for p in anchor_attention.parameters())
)

Input anchors: torch.Size([128, 16, 384])
Output anchors: torch.Size([128, 16, 384])
Parameters: 591360


# Anchor → Patch Cross-Attention

In [26]:
class PatchToAnchorAttention(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        num_heads=6,
        dropout=0.0
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_projection = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.kv_projection = nn.Linear(
            embed_dim,
            embed_dim * 2
        )

        self.output_projection = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, patches, anchors):
        batch_size = patches.shape[0]
        num_patches = patches.shape[1]
        num_anchors = anchors.shape[1]

        q = self.q_projection(anchors)

        kv = self.kv_projection(patches)

        k, v = kv.chunk(2, dim=-1)

        q = q.view(
            batch_size,
            num_anchors,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            num_patches,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            num_patches,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention_weights = attention_scores.softmax(
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(
            1,
            2
        ).reshape(
            batch_size,
            num_anchors,
            self.embed_dim
        )

        output = self.output_projection(output)
        output = self.projection_dropout(output)

        return output


patch_to_anchor = PatchToAnchorAttention(
    embed_dim=384,
    num_heads=6
).to(DEVICE)

with torch.no_grad():
    corrected_anchor_update = patch_to_anchor(
        patch_tokens_flat,
        initial_anchors
    )

print("Patches:", patch_tokens_flat.shape)
print("Anchors:", initial_anchors.shape)
print("Anchor update:", corrected_anchor_update.shape)
print(
    "Parameters:",
    sum(p.numel() for p in patch_to_anchor.parameters())
)

Patches: torch.Size([128, 196, 384])
Anchors: torch.Size([128, 16, 384])
Anchor update: torch.Size([128, 16, 384])
Parameters: 591360


# Anchor Block

In [27]:
class AnchorTransformerBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        num_heads=6,
        num_anchors=16,
        mlp_ratio=4.0,
        window_size=7,
        dropout=0.0
    ):
        super().__init__()

        self.patch_norm_local = nn.LayerNorm(embed_dim)
        self.patch_norm_global = nn.LayerNorm(embed_dim)
        self.anchor_norm = nn.LayerNorm(embed_dim)
        self.ffn_norm = nn.LayerNorm(embed_dim)

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=window_size,
            dropout=dropout
        )

        self.patch_to_anchor = PatchToAnchorAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.anchor_attention = AnchorSelfAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.anchor_to_patch = AnchorToPatchAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_anchors=num_anchors,
            dropout=dropout
        )

        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.beta = nn.Parameter(torch.tensor(1.0))

        hidden_dim = int(embed_dim * mlp_ratio)

        self.ffn = FeedForward(
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            dropout=dropout
        )

    def forward(self, patches, anchors):
        batch_size, num_patches, embed_dim = patches.shape

        local_input = self.patch_norm_local(patches)

        local_input = local_input.view(
            batch_size,
            14,
            14,
            embed_dim
        )

        local_context = self.local_attention(
            local_input
        )

        local_context = local_context.view(
            batch_size,
            num_patches,
            embed_dim
        )

        anchor_input = self.patch_norm_global(patches)

        anchor_update = self.patch_to_anchor(
            anchor_input,
            anchors
        )

        anchors = anchors + anchor_update

        anchor_context = self.anchor_attention(
            self.anchor_norm(anchors)
        )

        anchors = anchors + anchor_context

        global_context = self.anchor_to_patch(
            self.patch_norm_global(patches),
            self.anchor_norm(anchors)
        )

        patches = (
            patches
            + self.alpha * local_context
            + self.beta * global_context
        )

        patches = patches + self.ffn(
            self.ffn_norm(patches)
        )

        return patches, anchors


anchor_block = AnchorTransformerBlock(
    embed_dim=384,
    num_heads=6,
    num_anchors=16,
    window_size=7
).to(DEVICE)

with torch.no_grad():
    block_patches, block_anchors = anchor_block(
        patch_tokens_flat,
        initial_anchors
    )

print("Output patches:", block_patches.shape)
print("Output anchors:", block_anchors.shape)
print("Alpha:", anchor_block.alpha.item())
print("Beta:", anchor_block.beta.item())
print(
    "Block parameters:",
    sum(p.numel() for p in anchor_block.parameters())
)

Output patches: torch.Size([128, 196, 384])
Output anchors: torch.Size([128, 16, 384])
Alpha: 1.0
Beta: 1.0
Block parameters: 3550082


# RART Model

In [28]:
class RART(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        depth=6,
        num_heads=6,
        num_anchors=16,
        mlp_ratio=4.0,
        window_size=7
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            num_patches=self.patch_embed.num_patches,
            embed_dim=embed_dim
        )

        self.anchor_tokens = nn.Parameter(
            torch.randn(1, num_anchors, embed_dim) * 0.02
        )

        self.blocks = nn.ModuleList([
            AnchorTransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                num_anchors=num_anchors,
                mlp_ratio=mlp_ratio,
                window_size=window_size
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):
        x = self.patch_embed(x)

        x = self.input_embedding(x)

        cls_token = x[:, :1]
        patches = x[:, 1:]

        anchors = self.anchor_tokens.expand(
            x.shape[0],
            -1,
            -1
        )

        for block in self.blocks:
            patches, anchors = block(
                patches,
                anchors
            )

        cls_token = cls_token + patches.mean(
            dim=1,
            keepdim=True
        )

        cls_token = self.norm(
            cls_token
        )

        return self.head(
            cls_token[:, 0]
        )


rart = RART(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_classes=NUM_CLASSES,
    embed_dim=384,
    depth=6,
    num_heads=6,
    num_anchors=16,
    window_size=7
).to(DEVICE)

total_params = sum(
    p.numel() for p in rart.parameters()
)

print("Parameters:", total_params)

with torch.no_grad():
    rart_logits = rart(images_gpu)

print("Input:", images_gpu.shape)
print("Logits:", rart_logits.shape)

Parameters: 21717232
Input: torch.Size([128, 3, 224, 224])
Logits: torch.Size([128, 100])


RART BenchMark

In [29]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

avg_time_rart, peak_memory_rart = benchmark_model(
    rart,
    images_gpu
)

print("Parameters:", sum(p.numel() for p in rart.parameters()))
print(
    "Average inference time:",
    round(avg_time_rart * 1000, 3),
    "ms"
)
print(
    "Peak GPU memory:",
    round(peak_memory_rart / 1024**2, 2),
    "MB"
)
print(
    "Throughput:",
    round(128 / avg_time_rart, 2),
    "images/sec"
)

print()
print("ViT baseline:")
print("Parameters: 11057380")
print("Inference: 201.345 ms")
print("Peak memory: 2039.25 MB")
print("Throughput: 635.73 images/sec")

Parameters: 21717232
Average inference time: 291.56 ms
Peak GPU memory: 2604.94 MB
Throughput: 439.02 images/sec

ViT baseline:
Parameters: 11057380
Inference: 201.345 ms
Peak memory: 2039.25 MB
Throughput: 635.73 images/sec


# parameter budget mathematically

In [30]:
embed_dim = 384
anchor_dim = 128
num_anchors = 16
num_patches = 196
num_heads = 6

local_attention_params = (
    3 * embed_dim * embed_dim
    + embed_dim * embed_dim
    + 4 * embed_dim
)

anchor_projection_params = (
    embed_dim * anchor_dim
    + embed_dim * anchor_dim * 2
    + anchor_dim * embed_dim
)

anchor_attention_params = (
    3 * anchor_dim * anchor_dim
    + anchor_dim * anchor_dim
    + 4 * anchor_dim
)

print("Full local attention:", local_attention_params)
print("Compressed anchor pathway:", anchor_projection_params)
print("Compressed anchor attention:", anchor_attention_params)

print(
    "Current RART:",
    sum(p.numel() for p in rart.parameters())
)

print(
    "ViT baseline:",
    sum(p.numel() for p in vit.parameters())
)

Full local attention: 591360
Compressed anchor pathway: 196608
Compressed anchor attention: 66048
Current RART: 21717232
ViT baseline: 11057380


In [ ]:
# 384-D patches
#       │
#       ├── local attention ───────────► 384-D
#       │
#       └── compress ─► 128-D anchors
#                          │
#                          ▼
#                    global attention
#                          │
#                          ▼
#                     128-D anchors
#                          │
#                     expand back
#                          ▼
#                       384-D 

# Compressed Patch → Anchor

In [31]:
class CompressedPatchToAnchor(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=4,
        dropout=0.0
    ):
        super().__init__()

        assert anchor_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = num_heads
        self.head_dim = anchor_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.patch_projection = nn.Linear(
            embed_dim,
            anchor_dim
        )

        self.q_projection = nn.Linear(
            anchor_dim,
            anchor_dim
        )

        self.kv_projection = nn.Linear(
            anchor_dim,
            anchor_dim * 2
        )

        self.output_projection = nn.Linear(
            anchor_dim,
            anchor_dim
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, patches, anchors):
        batch_size = patches.shape[0]
        num_patches = patches.shape[1]
        num_anchors = anchors.shape[1]

        compressed_patches = self.patch_projection(
            patches
        )

        q = self.q_projection(anchors)

        kv = self.kv_projection(
            compressed_patches
        )

        k, v = kv.chunk(2, dim=-1)

        q = q.view(
            batch_size,
            num_anchors,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            num_patches,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            num_patches,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention_weights = attention_scores.softmax(
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(
            1,
            2
        ).reshape(
            batch_size,
            num_anchors,
            self.anchor_dim
        )

        output = self.output_projection(output)
        output = self.projection_dropout(output)

        return output

# Compressed Anchor → Patch

In [32]:
class CompressedAnchorToPatch(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=4,
        dropout=0.0
    ):
        super().__init__()

        assert anchor_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = num_heads
        self.head_dim = anchor_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.patch_projection = nn.Linear(
            embed_dim,
            anchor_dim
        )

        self.q_projection = nn.Linear(
            anchor_dim,
            anchor_dim
        )

        self.kv_projection = nn.Linear(
            anchor_dim,
            anchor_dim * 2
        )

        self.output_projection = nn.Linear(
            anchor_dim,
            embed_dim
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, patches, anchors):
        batch_size = patches.shape[0]
        num_patches = patches.shape[1]
        num_anchors = anchors.shape[1]

        compressed_patches = self.patch_projection(
            patches
        )

        q = self.q_projection(
            compressed_patches
        )

        kv = self.kv_projection(anchors)

        k, v = kv.chunk(2, dim=-1)

        q = q.view(
            batch_size,
            num_patches,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            num_anchors,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            num_anchors,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention_weights = attention_scores.softmax(
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(
            1,
            2
        ).reshape(
            batch_size,
            num_patches,
            self.anchor_dim
        )

        output = self.output_projection(output)
        output = self.projection_dropout(output)

        return output

In [33]:
class CompressedAnchorSelfAttention(nn.Module):
    def __init__(
        self,
        anchor_dim=128,
        num_heads=4,
        dropout=0.0
    ):
        super().__init__()

        assert anchor_dim % num_heads == 0

        self.anchor_dim = anchor_dim
        self.num_heads = num_heads
        self.head_dim = anchor_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(
            anchor_dim,
            anchor_dim * 3
        )

        self.projection = nn.Linear(
            anchor_dim,
            anchor_dim
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, num_anchors, anchor_dim = x.shape

        qkv = self.qkv(x)

        qkv = qkv.reshape(
            batch_size,
            num_anchors,
            3,
            self.num_heads,
            self.head_dim
        )

        qkv = qkv.permute(
            2,
            0,
            3,
            1,
            4
        )

        q, k, v = qkv.unbind(0)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention_weights = attention_scores.softmax(
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(
            1,
            2
        ).reshape(
            batch_size,
            num_anchors,
            anchor_dim
        )

        output = self.projection(output)
        output = self.projection_dropout(output)

        return output


compressed_anchor_attention = CompressedAnchorSelfAttention(
    anchor_dim=128,
    num_heads=4
).to(DEVICE)

test_anchors = torch.randn(
    128,
    16,
    128,
    device=DEVICE
)

with torch.no_grad():
    test_anchor_output = compressed_anchor_attention(
        test_anchors
    )

print("Input:", test_anchors.shape)
print("Output:", test_anchor_output.shape)
print(
    "Parameters:",
    sum(p.numel() for p in compressed_anchor_attention.parameters())
)

Input: torch.Size([128, 16, 128])
Output: torch.Size([128, 16, 128])
Parameters: 66048


In [ ]:
# 384-D patches
#    │
#    ├── local 7×7 attention ───────────┐
#    │                                  │
#    └── 384 → 128 → anchor attention   │
#                     ↓                 │
#                   128 → 384 ──────────┘
#                          ↓
#                  residual fusion

# Compressed RART Block

In [34]:
class CompressedRARTBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=4.0,
        window_size=7,
        dropout=0.0
    ):
        super().__init__()

        self.patch_norm_local = nn.LayerNorm(embed_dim)
        self.patch_norm_global = nn.LayerNorm(embed_dim)
        self.anchor_norm = nn.LayerNorm(anchor_dim)
        self.ffn_norm = nn.LayerNorm(embed_dim)

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=window_size,
            dropout=dropout
        )

        self.patch_to_anchor = CompressedPatchToAnchor(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads,
            dropout=dropout
        )

        self.anchor_attention = CompressedAnchorSelfAttention(
            anchor_dim=anchor_dim,
            num_heads=anchor_heads,
            dropout=dropout
        )

        self.anchor_to_patch = CompressedAnchorToPatch(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads,
            dropout=dropout
        )

        self.alpha = nn.Parameter(
            torch.tensor(1.0)
        )

        self.beta = nn.Parameter(
            torch.tensor(1.0)
        )

        hidden_dim = int(
            embed_dim * mlp_ratio
        )

        self.ffn = FeedForward(
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            dropout=dropout
        )

    def forward(self, patches, anchors):
        batch_size, num_patches, embed_dim = patches.shape

        local_input = self.patch_norm_local(
            patches
        )

        local_input = local_input.view(
            batch_size,
            14,
            14,
            embed_dim
        )

        local_context = self.local_attention(
            local_input
        )

        local_context = local_context.view(
            batch_size,
            num_patches,
            embed_dim
        )

        anchor_input = self.patch_norm_global(
            patches
        )

        anchor_update = self.patch_to_anchor(
            anchor_input,
            anchors
        )

        anchors = anchors + anchor_update

        anchor_context = self.anchor_attention(
            self.anchor_norm(anchors)
        )

        anchors = anchors + anchor_context

        global_context = self.anchor_to_patch(
            self.patch_norm_global(patches),
            self.anchor_norm(anchors)
        )

        patches = (
            patches
            + self.alpha * local_context
            + self.beta * global_context
        )

        patches = patches + self.ffn(
            self.ffn_norm(patches)
        )

        return patches, anchors


compressed_block = CompressedRARTBlock(
    embed_dim=384,
    anchor_dim=128,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    window_size=7
).to(DEVICE)

test_patch_tokens = patch_tokens_flat

test_anchor_tokens = torch.randn(
    128,
    16,
    128,
    device=DEVICE
)

with torch.no_grad():
    test_patch_output, test_anchor_output = compressed_block(
        test_patch_tokens,
        test_anchor_tokens
    )

print("Patch input:", test_patch_tokens.shape)
print("Patch output:", test_patch_output.shape)
print("Anchor input:", test_anchor_tokens.shape)
print("Anchor output:", test_anchor_output.shape)
print("Alpha:", compressed_block.alpha.item())
print("Beta:", compressed_block.beta.item())
print(
    "Block parameters:",
    sum(p.numel() for p in compressed_block.parameters())
)


Patch input: torch.Size([128, 196, 384])
Patch output: torch.Size([128, 196, 384])
Anchor input: torch.Size([128, 16, 128])
Anchor output: torch.Size([128, 16, 128])
Alpha: 1.0
Beta: 1.0
Block parameters: 2105218


# Full Compressed RART

In [35]:
class CompressedRART(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        anchor_dim=128,
        depth=6,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=4.0,
        window_size=7
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            num_patches=self.patch_embed.num_patches,
            embed_dim=embed_dim
        )

        self.anchor_tokens = nn.Parameter(
            torch.randn(
                1,
                num_anchors,
                anchor_dim
            ) * 0.02
        )

        self.blocks = nn.ModuleList([
            CompressedRARTBlock(
                embed_dim=embed_dim,
                anchor_dim=anchor_dim,
                num_heads=num_heads,
                anchor_heads=anchor_heads,
                num_anchors=num_anchors,
                mlp_ratio=mlp_ratio,
                window_size=window_size
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):
        x = self.patch_embed(x)

        x = self.input_embedding(x)

        cls_token = x[:, :1]
        patches = x[:, 1:]

        anchors = self.anchor_tokens.expand(
            x.shape[0],
            -1,
            -1
        )

        for block in self.blocks:
            patches, anchors = block(
                patches,
                anchors
            )

        cls_token = cls_token + patches.mean(
            dim=1,
            keepdim=True
        )

        cls_token = self.norm(
            cls_token
        )

        return self.head(
            cls_token[:, 0]
        )


compressed_rart = CompressedRART(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_classes=NUM_CLASSES,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    window_size=7
).to(DEVICE)

compressed_params = sum(
    p.numel() for p in compressed_rart.parameters()
)

print("Compressed RART parameters:", compressed_params)

with torch.no_grad():
    compressed_logits = compressed_rart(
        images_gpu
    )

print("Input:", images_gpu.shape)
print("Logits:", compressed_logits.shape)

Compressed RART parameters: 13043952
Input: torch.Size([128, 3, 224, 224])
Logits: torch.Size([128, 100])


# Benchmark Compressed RART

In [36]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

avg_time_compressed, peak_memory_compressed = benchmark_model(
    compressed_rart,
    images_gpu
)

print("Compressed RART")
print("Parameters:", compressed_params)
print(
    "Average inference time:",
    round(avg_time_compressed * 1000, 3),
    "ms"
)
print(
    "Peak GPU memory:",
    round(peak_memory_compressed / 1024**2, 2),
    "MB"
)
print(
    "Throughput:",
    round(128 / avg_time_compressed, 2),
    "images/sec"
)

print()
print("ViT baseline")
print("Parameters: 11057380")
print("Inference: 201.345 ms")
print("Peak memory: 2039.25 MB")
print("Throughput: 635.73 images/sec")

Compressed RART
Parameters: 13043952
Average inference time: 239.618 ms
Peak GPU memory: 2695.29 MB
Throughput: 534.18 images/sec

ViT baseline
Parameters: 11057380
Inference: 201.345 ms
Peak memory: 2039.25 MB
Throughput: 635.73 images/sec



**ViT:6 × (attention + 4D FFN)**

**RART:6 × (local attention + anchor pathway + 4D FFN)**


In [38]:
def count_parameters(module):
    return sum(
        p.numel()
        for p in module.parameters()
    )


sample_block = compressed_rart.blocks[0]

print("Patch embedding:", count_parameters(compressed_rart.patch_embed))
print("Input embedding:", count_parameters(compressed_rart.input_embedding))
print(
    "Anchor tokens:",
    compressed_rart.anchor_tokens.numel()
)
print("One RART block:", count_parameters(sample_block))
print("RART head:", count_parameters(compressed_rart.head))

print()
print("ViT total:", count_parameters(vit))
print("RART total:", count_parameters(compressed_rart))

Patch embedding: 295296
Input embedding: 76032
Anchor tokens: 2048
One RART block: 2105218
RART head: 38500

ViT total: 11057380
RART total: 13043952


# Parameter-matched RART

In [39]:
compressed_rart = CompressedRART(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_classes=NUM_CLASSES,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    mlp_ratio=3.0,
    window_size=7
).to(DEVICE)

matched_params = sum(
    p.numel()
    for p in compressed_rart.parameters()
)

print("ViT parameters:", sum(p.numel() for p in vit.parameters()))
print("RART parameters:", matched_params)
print(
    "Parameter difference:",
    matched_params - sum(p.numel() for p in vit.parameters())
)
print(
    "Parameter ratio:",
    round(
        matched_params / sum(p.numel() for p in vit.parameters()),
        3
    )
)

with torch.no_grad():
    matched_logits = compressed_rart(images_gpu)

print("Logits:", matched_logits.shape)

ViT parameters: 11057380
RART parameters: 11272176
Parameter difference: 214796
Parameter ratio: 1.019
Logits: torch.Size([128, 100])


# Parameter-matched benchmark

In [40]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

avg_time_matched, peak_memory_matched = benchmark_model(
    compressed_rart,
    images_gpu
)

print("Parameter-matched RART")
print("Parameters:", matched_params)
print(
    "Average inference time:",
    round(avg_time_matched * 1000, 3),
    "ms"
)
print(
    "Peak GPU memory:",
    round(peak_memory_matched / 1024**2, 2),
    "MB"
)
print(
    "Throughput:",
    round(128 / avg_time_matched, 2),
    "images/sec"
)

print()
print("ViT baseline")
print("Parameters: 11057380")
print("Inference: 201.345 ms")
print("Peak memory: 2039.25 MB")
print("Throughput: 635.73 images/sec")

Parameter-matched RART
Parameters: 11272176
Average inference time: 212.667 ms
Peak GPU memory: 2619.68 MB
Throughput: 601.88 images/sec

ViT baseline
Parameters: 11057380
Inference: 201.345 ms
Peak memory: 2039.25 MB
Throughput: 635.73 images/sec


# Train RART for 10 epochs

In [41]:
criterion_rart = nn.CrossEntropyLoss()

optimizer_rart = torch.optim.AdamW(
    compressed_rart.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler_rart = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_rart,
    T_max=10
)

scaler_rart = torch.amp.GradScaler("cuda")

rart_history = []

for epoch in range(1, 11):
    start_time = time.perf_counter()

    train_loss, train_acc = train_one_epoch_amp(
        compressed_rart,
        train_loader,
        optimizer_rart,
        criterion_rart,
        DEVICE
    )

    test_loss, test_acc = evaluate_amp(
        compressed_rart,
        test_loader,
        criterion_rart,
        DEVICE
    )

    scheduler_rart.step()

    elapsed = time.perf_counter() - start_time

    rart_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "lr": optimizer_rart.param_groups[0]["lr"],
        "time": elapsed / 60
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}% | "
        f"Time: {elapsed / 60:.2f} min"
    )

Epoch 01 | Train Loss: 3.8761 | Train Acc: 10.45% | Test Loss: 3.5047 | Test Acc: 16.78% | Time: 2.03 min
Epoch 02 | Train Loss: 3.3218 | Train Acc: 19.44% | Test Loss: 3.1823 | Test Acc: 22.55% | Time: 2.04 min
Epoch 03 | Train Loss: 3.0375 | Train Acc: 24.49% | Test Loss: 2.9555 | Test Acc: 26.18% | Time: 2.04 min
Epoch 04 | Train Loss: 2.8197 | Train Acc: 28.99% | Test Loss: 2.8044 | Test Acc: 29.83% | Time: 2.03 min
Epoch 05 | Train Loss: 2.6090 | Train Acc: 33.10% | Test Loss: 2.6711 | Test Acc: 32.65% | Time: 2.04 min
Epoch 06 | Train Loss: 2.3963 | Train Acc: 37.68% | Test Loss: 2.5570 | Test Acc: 34.40% | Time: 2.04 min
Epoch 07 | Train Loss: 2.1785 | Train Acc: 42.57% | Test Loss: 2.3880 | Test Acc: 38.58% | Time: 2.03 min
Epoch 08 | Train Loss: 1.9817 | Train Acc: 47.31% | Test Loss: 2.3124 | Test Acc: 40.04% | Time: 2.04 min
Epoch 09 | Train Loss: 1.8219 | Train Acc: 51.26% | Test Loss: 2.2537 | Test Acc: 41.81% | Time: 2.04 min
Epoch 10 | Train Loss: 1.7279 | Train Acc: 54.

# Saving Results

In [42]:
import pandas as pd

vit_results = pd.DataFrame(history)
rart_results = pd.DataFrame(rart_history)

print("ViT results:")
display(vit_results)

print()
print("RART results:")
display(rart_results)

print()
print("Best ViT test accuracy:", round(vit_results["test_acc"].max(), 2), "%")
print("Best RART test accuracy:", round(rart_results["test_acc"].max(), 2), "%")

ViT results:


,epoch,train_loss,train_acc,test_loss,test_acc,lr,time
0,1,3.806857,11.392,3.474935,16.53,0.000293,1.880135
1,2,3.247924,20.392,3.106935,23.41,0.000271,1.871281
2,3,2.929932,26.662,2.868539,28.23,0.000238,1.893943
3,4,2.660654,31.876,2.647994,32.94,0.000196,1.891982
4,5,2.385419,37.424,2.487374,36.21,0.000150,1.872610
5,6,2.117965,43.630,2.365631,39.31,0.000104,1.870105
6,7,1.840305,50.148,2.240615,42.22,0.000062,1.873968
7,8,1.571275,57.092,2.167447,43.75,0.000029,1.888716
8,9,1.346373,63.392,2.126754,45.14,0.000007,1.893687
9,10,1.212368,67.256,2.113141,45.44,0.000000,1.887354



RART results:


,epoch,train_loss,train_acc,test_loss,test_acc,lr,time
0,1,3.876069,10.454,3.504747,16.78,0.000293,2.033237
1,2,3.321829,19.440,3.182335,22.55,0.000271,2.035497
2,3,3.037478,24.486,2.955486,26.18,0.000238,2.037449
3,4,2.819665,28.988,2.804398,29.83,0.000196,2.030901
4,5,2.609048,33.096,2.671125,32.65,0.000150,2.036761
5,6,2.396320,37.678,2.556952,34.40,0.000104,2.042474
6,7,2.178470,42.572,2.387958,38.58,0.000062,2.025449
7,8,1.981704,47.314,2.312362,40.04,0.000029,2.036089
8,9,1.821938,51.260,2.253686,41.81,0.000007,2.038172
9,10,1.727866,54.086,2.225667,42.21,0.000000,2.037432



Best ViT test accuracy: 45.44 %
Best RART test accuracy: 42.21 %


# Changing Achors to 32

In [43]:
rart_32 = CompressedRART(
    image_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_classes=NUM_CLASSES,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=32,
    mlp_ratio=3.0,
    window_size=7
).to(DEVICE)

params_32 = sum(
    p.numel() for p in rart_32.parameters()
)

print("32-anchor RART parameters:", params_32)

with torch.no_grad():
    logits_32 = rart_32(images_gpu)

print("Logits:", logits_32.shape)

32-anchor RART parameters: 11274224
Logits: torch.Size([128, 100])


# Benchmark 32 Anchors

In [44]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

avg_time_32, peak_memory_32 = benchmark_model(
    rart_32,
    images_gpu
)

print("32-anchor RART")
print("Parameters:", params_32)
print(
    "Average inference time:",
    round(avg_time_32 * 1000, 3),
    "ms"
)
print(
    "Peak GPU memory:",
    round(peak_memory_32 / 1024**2, 2),
    "MB"
)
print(
    "Throughput:",
    round(128 / avg_time_32, 2),
    "images/sec"
)

32-anchor RART
Parameters: 11274224
Average inference time: 215.096 ms
Peak GPU memory: 2802.46 MB
Throughput: 595.08 images/sec


# Train 32 RART

In [45]:
criterion_32 = nn.CrossEntropyLoss()

optimizer_32 = torch.optim.AdamW(
    rart_32.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler_32 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_32,
    T_max=10
)

scaler_32 = torch.amp.GradScaler("cuda")

rart_32_history = []

for epoch in range(1, 11):
    start_time = time.perf_counter()

    train_loss, train_acc = train_one_epoch_amp(
        rart_32,
        train_loader,
        optimizer_32,
        criterion_32,
        DEVICE
    )

    test_loss, test_acc = evaluate_amp(
        rart_32,
        test_loader,
        criterion_32,
        DEVICE
    )

    scheduler_32.step()

    elapsed = time.perf_counter() - start_time

    rart_32_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "lr": optimizer_32.param_groups[0]["lr"],
        "time": elapsed / 60
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}% | "
        f"Time: {elapsed / 60:.2f} min"
    )

Epoch 01 | Train Loss: 3.8854 | Train Acc: 10.33% | Test Loss: 3.5110 | Test Acc: 16.20% | Time: 2.07 min
Epoch 02 | Train Loss: 3.3391 | Train Acc: 19.11% | Test Loss: 3.2221 | Test Acc: 21.63% | Time: 2.08 min
Epoch 03 | Train Loss: 3.0889 | Train Acc: 23.67% | Test Loss: 3.0749 | Test Acc: 24.66% | Time: 2.08 min
Epoch 04 | Train Loss: 2.8893 | Train Acc: 27.61% | Test Loss: 2.9093 | Test Acc: 27.44% | Time: 2.09 min
Epoch 05 | Train Loss: 2.6946 | Train Acc: 31.41% | Test Loss: 2.7766 | Test Acc: 30.39% | Time: 2.08 min
Epoch 06 | Train Loss: 2.4880 | Train Acc: 35.80% | Test Loss: 2.6225 | Test Acc: 33.30% | Time: 2.10 min
Epoch 07 | Train Loss: 2.2731 | Train Acc: 40.55% | Test Loss: 2.5189 | Test Acc: 35.47% | Time: 2.09 min
Epoch 08 | Train Loss: 2.0765 | Train Acc: 45.21% | Test Loss: 2.4387 | Test Acc: 37.60% | Time: 2.11 min
Epoch 09 | Train Loss: 1.9163 | Train Acc: 49.22% | Test Loss: 2.3791 | Test Acc: 39.44% | Time: 2.09 min
Epoch 10 | Train Loss: 1.8152 | Train Acc: 51.

# current RART information flow

In [46]:
print("Baseline:")
print("Input tokens:", 197)
print("Transformer tokens:", 197)
print("CLS participates in attention: Yes")

print("\nRART:")
print("Input tokens:", 197)
print("Transformer patch tokens:", 196)
print("Anchor tokens:", 32)
print("CLS participates in block attention: No")
print("Final classifier: CLS + mean(patches)")

Baseline:
Input tokens: 197
Transformer tokens: 197
CLS participates in attention: Yes

RART:
Input tokens: 197
Transformer patch tokens: 196
Anchor tokens: 32
CLS participates in block attention: No
Final classifier: CLS + mean(patches)


# create a mean-pooled ViT baseline

In [47]:
class ViTMeanPool(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        num_classes=100,
        embed_dim=384,
        depth=6,
        num_heads=6,
        mlp_ratio=3.0
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=3,
            embed_dim=embed_dim
        )

        num_patches = (image_size // patch_size) ** 2

        self.pos_embed = nn.Parameter(
            torch.randn(1, num_patches, embed_dim) * 0.02
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        x = x.mean(dim=1)

        return self.head(x)


vit_mean = ViTMeanPool(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    depth=6,
    num_heads=6,
    mlp_ratio=3.0
).to(DEVICE)

print(f"Parameters: {sum(p.numel() for p in vit_mean.parameters()):,}")

dummy = torch.randn(128, 3, 224, 224, device=DEVICE)

with torch.no_grad():
    logits = vit_mean(dummy)

print("Logits shape:", logits.shape)

Parameters: 9,284,836
Logits shape: torch.Size([128, 100])


# matching MLP ratio

In [48]:
target_params = sum(p.numel() for p in rart_32.parameters())

for ratio in [3.0, 3.25, 3.5, 3.75, 4.0, 4.25, 4.5]:
    candidate = ViTMeanPool(
        image_size=224,
        patch_size=16,
        num_classes=100,
        embed_dim=384,
        depth=6,
        num_heads=6,
        mlp_ratio=ratio
    )

    params = sum(p.numel() for p in candidate.parameters())

    print(
        f"MLP ratio {ratio:.2f} | "
        f"Parameters: {params:,} | "
        f"Difference vs RART-32: {params - target_params:+,}"
    )

MLP ratio 3.00 | Parameters: 9,284,836 | Difference vs RART-32: -1,989,388
MLP ratio 3.25 | Parameters: 9,727,780 | Difference vs RART-32: -1,546,444
MLP ratio 3.50 | Parameters: 10,170,724 | Difference vs RART-32: -1,103,500
MLP ratio 3.75 | Parameters: 10,613,668 | Difference vs RART-32: -660,556
MLP ratio 4.00 | Parameters: 11,056,612 | Difference vs RART-32: -217,612
MLP ratio 4.25 | Parameters: 11,499,556 | Difference vs RART-32: +225,332
MLP ratio 4.50 | Parameters: 11,942,500 | Difference vs RART-32: +668,276


In [49]:
vit_mean = ViTMeanPool(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    depth=6,
    num_heads=6,
    mlp_ratio=4.0
).to(DEVICE)

criterion_mean = nn.CrossEntropyLoss()

optimizer_mean = torch.optim.AdamW(
    vit_mean.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler_mean = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_mean,
    T_max=10
)

scaler_mean = torch.amp.GradScaler("cuda")

vit_mean_history = []

for epoch in range(1, 11):
    start_time = time.perf_counter()

    train_loss, train_acc = train_one_epoch_amp(
        vit_mean,
        train_loader,
        optimizer_mean,
        criterion_mean,
        DEVICE
    )

    test_loss, test_acc = evaluate_amp(
        vit_mean,
        test_loader,
        criterion_mean,
        DEVICE
    )

    scheduler_mean.step()

    elapsed = time.perf_counter() - start_time

    vit_mean_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "lr": optimizer_mean.param_groups[0]["lr"],
        "time": elapsed / 60
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}% | "
        f"Time: {elapsed / 60:.2f} min"
    )

Epoch 01 | Train Loss: 3.8682 | Train Acc: 10.50% | Test Loss: 3.5834 | Test Acc: 14.54% | Time: 1.86 min
Epoch 02 | Train Loss: 3.3727 | Train Acc: 18.40% | Test Loss: 3.2618 | Test Acc: 20.63% | Time: 1.88 min
Epoch 03 | Train Loss: 3.0875 | Train Acc: 23.73% | Test Loss: 2.9851 | Test Acc: 26.36% | Time: 1.88 min
Epoch 04 | Train Loss: 2.8292 | Train Acc: 28.86% | Test Loss: 2.8800 | Test Acc: 27.72% | Time: 1.88 min
Epoch 05 | Train Loss: 2.5692 | Train Acc: 34.14% | Test Loss: 2.5828 | Test Acc: 34.23% | Time: 1.88 min
Epoch 06 | Train Loss: 2.3151 | Train Acc: 39.29% | Test Loss: 2.4255 | Test Acc: 37.71% | Time: 1.88 min
Epoch 07 | Train Loss: 2.0716 | Train Acc: 44.74% | Test Loss: 2.3031 | Test Acc: 40.11% | Time: 1.88 min
Epoch 08 | Train Loss: 1.8538 | Train Acc: 49.89% | Test Loss: 2.2093 | Test Acc: 42.45% | Time: 1.88 min
Epoch 09 | Train Loss: 1.6745 | Train Acc: 54.38% | Test Loss: 2.1542 | Test Acc: 43.89% | Time: 1.88 min
Epoch 10 | Train Loss: 1.5659 | Train Acc: 57.

# build 16-anchor, 384D diagnostic RART

In [51]:
import inspect

print(inspect.signature(RART))
print("\nSource:")
print(inspect.getsource(RART.__init__))

(image_size=224, patch_size=16, in_channels=3, num_classes=100, embed_dim=384, depth=6, num_heads=6, num_anchors=16, mlp_ratio=4.0, window_size=7)

Source:
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        depth=6,
        num_heads=6,
        num_anchors=16,
        mlp_ratio=4.0,
        window_size=7
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            num_patches=self.patch_embed.num_patches,
            embed_dim=embed_dim
        )

        self.anchor_tokens = nn.Parameter(
            torch.randn(1, num_anchors, embed_dim) * 0.02
        )

        self.blocks = nn.ModuleList([
            AnchorTransformerBlock(
                embed_dim=

In [53]:
block = rart_32.blocks[0]

print(block)
print("\nNamed modules:")
for name, module in block.named_modules():
    if name:
        print(f"{name}: {module.__class__.__name__}")

print("\nParameter shapes:")
for name, param in block.named_parameters():
    print(f"{name}: {tuple(param.shape)}")

CompressedRARTBlock(
  (patch_norm_local): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (patch_norm_global): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (anchor_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (ffn_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (local_attention): LocalWindowAttention(
    (qkv): Linear(in_features=384, out_features=1152, bias=True)
    (projection): Linear(in_features=384, out_features=384, bias=True)
    (attention_dropout): Dropout(p=0.0, inplace=False)
    (projection_dropout): Dropout(p=0.0, inplace=False)
  )
  (patch_to_anchor): CompressedPatchToAnchor(
    (patch_projection): Linear(in_features=384, out_features=128, bias=True)
    (q_projection): Linear(in_features=128, out_features=128, bias=True)
    (kv_projection): Linear(in_features=128, out_features=256, bias=True)
    (output_projection): Linear(in_features=128, out_features=128, bias=True)
    (attention_dropout): Dropout(p=0.0, inpla

In [54]:
print("RART-32 routing strengths")

for i, block in enumerate(rart_32.blocks):
    print(
        f"Block {i + 1}: "
        f"alpha = {block.alpha.item():.6f}, "
        f"beta = {block.beta.item():.6f}"
    )

RART-32 routing strengths
Block 1: alpha = 0.944160, beta = 0.950757
Block 2: alpha = 0.965371, beta = 0.894627
Block 3: alpha = 0.958211, beta = 0.866810
Block 4: alpha = 0.944023, beta = 0.885903
Block 5: alpha = 0.905842, beta = 0.895898
Block 6: alpha = 0.919454, beta = 0.916813


In [55]:
import inspect

print(inspect.signature(CompressedRARTBlock))

(embed_dim=384, anchor_dim=128, num_heads=6, anchor_heads=4, num_anchors=16, mlp_ratio=4.0, window_size=7, dropout=0.0)


In [56]:
wide_block = CompressedRARTBlock(
    embed_dim=384,
    anchor_dim=384,
    num_heads=6,
    anchor_heads=6,
    num_anchors=16,
    mlp_ratio=3.0,
    window_size=7
).to(DEVICE)

print(wide_block)

print(
    f"\nBlock parameters: "
    f"{sum(p.numel() for p in wide_block.parameters()):,}"
)

CompressedRARTBlock(
  (patch_norm_local): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (patch_norm_global): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (anchor_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (ffn_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (local_attention): LocalWindowAttention(
    (qkv): Linear(in_features=384, out_features=1152, bias=True)
    (projection): Linear(in_features=384, out_features=384, bias=True)
    (attention_dropout): Dropout(p=0.0, inplace=False)
    (projection_dropout): Dropout(p=0.0, inplace=False)
  )
  (patch_to_anchor): CompressedPatchToAnchor(
    (patch_projection): Linear(in_features=384, out_features=384, bias=True)
    (q_projection): Linear(in_features=384, out_features=384, bias=True)
    (kv_projection): Linear(in_features=384, out_features=768, bias=True)
    (output_projection): Linear(in_features=384, out_features=384, bias=True)
    (attention_dropout): Dropout(p=0.0, inpla

# Wide RART

In [57]:
class RARTWide(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        depth=6,
        num_heads=6,
        num_anchors=16,
        mlp_ratio=3.0,
        window_size=7
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            num_patches=self.patch_embed.num_patches,
            embed_dim=embed_dim
        )

        self.anchor_tokens = nn.Parameter(
            torch.randn(1, num_anchors, embed_dim) * 0.02
        )

        self.blocks = nn.ModuleList([
            CompressedRARTBlock(
                embed_dim=embed_dim,
                anchor_dim=embed_dim,
                num_heads=num_heads,
                anchor_heads=num_heads,
                num_anchors=num_anchors,
                mlp_ratio=mlp_ratio,
                window_size=window_size
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.input_embedding(x)

        patches = x[:, 1:]
        anchors = self.anchor_tokens.expand(x.size(0), -1, -1)

        for block in self.blocks:
            patches, anchors = block(patches, anchors)

        patches = self.norm(patches)
        pooled = patches.mean(dim=1)

        return self.head(pooled)


rart_wide = RARTWide(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    depth=6,
    num_heads=6,
    num_anchors=16,
    mlp_ratio=3.0,
    window_size=7
).to(DEVICE)

params = sum(p.numel() for p in rart_wide.parameters())

print(f"Parameters: {params:,}")

with torch.no_grad():
    logits = rart_wide(dummy)

print("Logits shape:", logits.shape)

Parameters: 21,719,536
Logits shape: torch.Size([128, 100])


# Wide RART Benchmark

In [58]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

wide_time, wide_memory = benchmark_model(
    rart_wide,
    dummy,
    warmup=10,
    runs=50
)

wide_throughput = dummy.shape[0] / wide_time

print(f"Average inference time: {wide_time * 1000:.3f} ms")
print(f"Peak GPU memory: {wide_memory / 1024**2:.2f} MB")
print(f"Throughput: {wide_throughput:.2f} images/s")

Average inference time: 300.632 ms
Peak GPU memory: 3291.60 MB
Throughput: 425.77 images/s


# measure anchor diversity

In [59]:
@torch.no_grad()
def measure_anchor_diversity(model, loader, device, num_batches=10):
    model.eval()

    similarities = []
    norms = []

    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= num_batches:
            break

        images = images.to(device)

        x = model.patch_embed(images)
        x = model.input_embedding(x)

        patches = x[:, 1:]
        anchors = model.anchor_tokens.expand(images.size(0), -1, -1)

        for block in model.blocks:
            patches, anchors = block(patches, anchors)

            normalized = F.normalize(anchors, dim=-1)
            sim = torch.matmul(
                normalized,
                normalized.transpose(-1, -2)
            )

            n = sim.size(-1)
            mask = ~torch.eye(
                n,
                device=device,
                dtype=torch.bool
            )

            similarities.append(sim[:, mask].mean().item())
            norms.append(anchors.norm(dim=-1).mean().item())

    similarities = torch.tensor(similarities).view(-1, len(model.blocks))
    norms = torch.tensor(norms).view(-1, len(model.blocks))

    return similarities.mean(dim=0), norms.mean(dim=0)


anchor_sim, anchor_norm = measure_anchor_diversity(
    rart_32,
    test_loader,
    DEVICE
)

for i in range(len(anchor_sim)):
    print(
        f"Block {i + 1}: "
        f"Mean anchor cosine similarity = {anchor_sim[i]:.4f} | "
        f"Mean norm = {anchor_norm[i]:.4f}"
    )

Block 1: Mean anchor cosine similarity = 0.9887 | Mean norm = 13.6643
Block 2: Mean anchor cosine similarity = 0.9865 | Mean norm = 16.1777
Block 3: Mean anchor cosine similarity = 0.9900 | Mean norm = 21.0188
Block 4: Mean anchor cosine similarity = 0.9920 | Mean norm = 23.8078
Block 5: Mean anchor cosine similarity = 0.9935 | Mean norm = 26.7162
Block 6: Mean anchor cosine similarity = 0.9950 | Mean norm = 30.9705


# effective anchor rank

In [60]:
@torch.no_grad()
def anchor_rank(model):
    anchors = model.anchor_tokens.squeeze(0)

    anchors = anchors - anchors.mean(dim=0, keepdim=True)

    singular_values = torch.linalg.svdvals(anchors)
    explained = singular_values.square()
    explained = explained / explained.sum()

    cumulative = torch.cumsum(explained, dim=0)

    rank_90 = int((cumulative < 0.90).sum().item()) + 1
    rank_95 = int((cumulative < 0.95).sum().item()) + 1
    rank_99 = int((cumulative < 0.99).sum().item()) + 1

    print(f"Number of anchors: {anchors.shape[0]}")
    print(f"Rank for 90% variance: {rank_90}")
    print(f"Rank for 95% variance: {rank_95}")
    print(f"Rank for 99% variance: {rank_99}")

    print("\nNormalized singular values:")
    print((singular_values / singular_values[0]).cpu().numpy())


anchor_rank(rart_32)

Number of anchors: 32
Rank for 90% variance: 23
Rank for 95% variance: 26
Rank for 99% variance: 30

Normalized singular values:
[1.0000000e+00 5.7432669e-01 4.8795983e-01 4.7558925e-01 4.5724994e-01
 4.3756422e-01 4.2908630e-01 4.1648746e-01 4.0667257e-01 4.0215397e-01
 3.9253229e-01 3.8747007e-01 3.7187523e-01 3.6356136e-01 3.5484976e-01
 3.5106018e-01 3.3412540e-01 3.2772446e-01 3.1167576e-01 3.0356678e-01
 2.9420114e-01 2.8245914e-01 2.7340126e-01 2.6848578e-01 2.5932443e-01
 2.5267491e-01 2.2293019e-01 2.2264175e-01 2.1497336e-01 2.0354278e-01
 1.9164146e-01 7.8095908e-08]


In [61]:
@torch.no_grad()
def measure_anchor_effective_rank(model, loader, device, num_batches=5):
    model.eval()

    all_ranks = [[] for _ in model.blocks]

    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= num_batches:
            break

        images = images.to(device)

        x = model.patch_embed(images)
        x = model.input_embedding(x)

        patches = x[:, 1:]
        anchors = model.anchor_tokens.expand(images.size(0), -1, -1)

        for i, block in enumerate(model.blocks):
            patches, anchors = block(patches, anchors)

            centered = anchors - anchors.mean(dim=1, keepdim=True)

            singular_values = torch.linalg.svdvals(centered)

            variance = singular_values.square()
            variance = variance / variance.sum(dim=-1, keepdim=True)

            cumulative = torch.cumsum(variance, dim=-1)

            rank_90 = (cumulative < 0.90).sum(dim=-1) + 1
            rank_95 = (cumulative < 0.95).sum(dim=-1) + 1
            rank_99 = (cumulative < 0.99).sum(dim=-1) + 1

            all_ranks[i].append([
                rank_90.float().mean().item(),
                rank_95.float().mean().item(),
                rank_99.float().mean().item()
            ])

    for i, ranks in enumerate(all_ranks):
        ranks = torch.tensor(ranks).mean(dim=0)

        print(
            f"Block {i + 1}: "
            f"Rank90={ranks[0]:.2f}, "
            f"Rank95={ranks[1]:.2f}, "
            f"Rank99={ranks[2]:.2f}"
        )


measure_anchor_effective_rank(
    rart_32,
    test_loader,
    DEVICE
)

Block 1: Rank90=1.77, Rank95=2.96, Rank99=11.61
Block 2: Rank90=1.81, Rank95=2.87, Rank99=10.46
Block 3: Rank90=1.85, Rank95=2.86, Rank99=10.28
Block 4: Rank90=1.84, Rank95=2.82, Rank99=10.03
Block 5: Rank90=1.82, Rank95=2.79, Rank99=9.83
Block 6: Rank90=1.80, Rank95=2.73, Rank99=9.55


In [62]:
@torch.no_grad()
def measure_patch_to_anchor_rank(model, loader, device, num_batches=5):
    model.eval()

    results = []

    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= num_batches:
            break

        images = images.to(device)

        x = model.patch_embed(images)
        x = model.input_embedding(x)

        patches = x[:, 1:]
        anchors = model.anchor_tokens.expand(images.size(0), -1, -1)

        for block in model.blocks:
            normalized_patches = block.patch_norm_global(patches)

            anchors_after = block.patch_to_anchor(
                normalized_patches,
                anchors
            )

            centered = anchors_after - anchors_after.mean(
                dim=1,
                keepdim=True
            )

            singular_values = torch.linalg.svdvals(centered)

            variance = singular_values.square()
            variance = variance / variance.sum(dim=-1, keepdim=True)

            cumulative = torch.cumsum(variance, dim=-1)

            rank_90 = (cumulative < 0.90).sum(dim=-1) + 1
            rank_95 = (cumulative < 0.95).sum(dim=-1) + 1
            rank_99 = (cumulative < 0.99).sum(dim=-1) + 1

            results.append([
                rank_90.float().mean().item(),
                rank_95.float().mean().item(),
                rank_99.float().mean().item()
            ])

            break

    results = torch.tensor(results).mean(dim=0)

    print(f"After Patch → Anchor:")
    print(f"Rank90: {results[0]:.2f}")
    print(f"Rank95: {results[1]:.2f}")
    print(f"Rank99: {results[2]:.2f}")


measure_patch_to_anchor_rank(
    rart_32,
    test_loader,
    DEVICE
)

After Patch → Anchor:
Rank90: 1.18
Rank95: 1.74
Rank99: 2.83


# measure patch→anchor attention entropy

In [63]:
print(inspect.signature(CompressedPatchToAnchor.forward))
print("\nAttributes:")
for name in vars(rart_32.blocks[0].patch_to_anchor):
    print(name)

(self, patches, anchors)

Attributes:
training
_parameters
_buffers
_non_persistent_buffers_set
_backward_pre_hooks
_backward_hooks
_is_full_backward_hook
_forward_hooks
_forward_hooks_with_kwargs
_forward_hooks_always_called
_forward_pre_hooks
_forward_pre_hooks_with_kwargs
_state_dict_hooks
_state_dict_pre_hooks
_load_state_dict_pre_hooks
_load_state_dict_post_hooks
_modules
embed_dim
anchor_dim
num_heads
head_dim
scale


In [64]:
@torch.no_grad()
def analyze_patch_to_anchor_attention(model, loader, device, num_batches=5):
    model.eval()

    block = model.blocks[0]
    module = block.patch_to_anchor

    all_entropy = []
    all_effective = []
    all_similarity = []

    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= num_batches:
            break

        images = images.to(device)

        x = model.patch_embed(images)
        x = model.input_embedding(x)

        patches = x[:, 1:]
        anchors = model.anchor_tokens.expand(images.size(0), -1, -1)

        patches_norm = block.patch_norm_global(patches)

        patch_proj = module.patch_projection(patches_norm)
        anchor_proj = module.q_projection(anchors)

        B, N, D = patch_proj.shape
        A = anchor_proj.shape[1]

        patch_proj = patch_proj.view(
            B, N, module.num_heads, module.head_dim
        ).transpose(1, 2)

        anchor_proj = anchor_proj.view(
            B, A, module.num_heads, module.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            anchor_proj,
            patch_proj.transpose(-2, -1)
        ) * module.scale

        attention = torch.softmax(scores, dim=-1)

        entropy = -(
            attention *
            torch.log(attention.clamp_min(1e-9))
        ).sum(dim=-1)

        effective_patches = entropy.exp()

        attention_mean = attention.mean(dim=1)

        normalized = F.normalize(
            attention_mean,
            dim=-1
        )

        similarity = torch.matmul(
            normalized,
            normalized.transpose(-1, -2)
        )

        A = similarity.shape[-1]

        mask = ~torch.eye(
            A,
            device=device,
            dtype=torch.bool
        )

        all_entropy.append(entropy.mean().item())
        all_effective.append(effective_patches.mean().item())
        all_similarity.append(similarity[:, mask].mean().item())

    print(f"Mean attention entropy: {sum(all_entropy) / len(all_entropy):.4f}")
    print(f"Effective patches attended: {sum(all_effective) / len(all_effective):.2f} / 196")
    print(f"Mean anchor attention-map similarity: {sum(all_similarity) / len(all_similarity):.4f}")


analyze_patch_to_anchor_attention(
    rart_32,
    test_loader,
    DEVICE
)

Mean attention entropy: 5.2720
Effective patches attended: 194.82 / 196
Mean anchor attention-map similarity: 0.9995


# 32 anchors are effectively doing the same global pooling operation.

# initialization mechanism

In [67]:
@torch.no_grad()
def initialize_spatial_anchors(model, images):
    x = model.patch_embed(images)
    x = model.input_embedding(x)

    patches = x[:, 1:]

    B, N, D = patches.shape
    grid_size = int(N ** 0.5)

    num_anchors = model.anchor_tokens.shape[1]

    factors = [
        (r, num_anchors // r)
        for r in range(1, num_anchors + 1)
        if num_anchors % r == 0
    ]

    rows, cols = min(
        factors,
        key=lambda x: abs(x[0] - x[1])
    )

    row_positions = torch.linspace(
        0,
        grid_size - 1,
        rows,
        device=images.device
    ).round().long()

    col_positions = torch.linspace(
        0,
        grid_size - 1,
        cols,
        device=images.device
    ).round().long()

    indices = []

    for r in row_positions:
        for c in col_positions:
            indices.append(r * grid_size + c)

    indices = torch.tensor(
        indices,
        device=images.device
    )

    spatial_anchors = patches[:, indices]

    spatial_anchors = model.blocks[0].patch_to_anchor.patch_projection(
        spatial_anchors
    )

    learned_anchors = model.anchor_tokens.expand(
        B, -1, -1
    )

    return spatial_anchors + learned_anchors


spatial_anchors = initialize_spatial_anchors(
    rart_32,
    sample_images
)

print("Spatial anchor shape:", spatial_anchors.shape)
print(
    "Mean norm:",
    spatial_anchors.norm(dim=-1).mean().item()
)

Spatial anchor shape: torch.Size([128, 32, 128])
Mean norm: 22.540016174316406


In [69]:
@torch.no_grad()
def spatial_anchor_attention_similarity(model, images):
    x = model.patch_embed(images)
    x = model.input_embedding(x)

    patches = x[:, 1:]

    B, N, D = patches.shape
    grid_size = int(N ** 0.5)

    num_anchors = model.anchor_tokens.shape[1]

    factors = [
        (r, num_anchors // r)
        for r in range(1, num_anchors + 1)
        if num_anchors % r == 0
    ]

    rows, cols = min(
        factors,
        key=lambda x: abs(x[0] - x[1])
    )

    row_positions = torch.linspace(
        0,
        grid_size - 1,
        rows,
        device=images.device
    ).round().long()

    col_positions = torch.linspace(
        0,
        grid_size - 1,
        cols,
        device=images.device
    ).round().long()

    indices = torch.tensor(
        [
            r * grid_size + c
            for r in row_positions
            for c in col_positions
        ],
        device=images.device
    )

    spatial_anchors = patches[:, indices]

    block = model.blocks[0]
    module = block.patch_to_anchor

    normalized_patches = block.patch_norm_global(patches)

    patch_proj = module.patch_projection(
        normalized_patches
    )

    anchor_proj = module.patch_projection(
        spatial_anchors
    )

    anchor_proj = module.q_projection(
        anchor_proj
    )

    patch_proj = patch_proj.view(
        B, N, module.num_heads, module.head_dim
    ).transpose(1, 2)

    anchor_proj = anchor_proj.view(
        B, num_anchors, module.num_heads, module.head_dim
    ).transpose(1, 2)

    scores = torch.matmul(
        anchor_proj,
        patch_proj.transpose(-2, -1)
    ) * module.scale

    attention = torch.softmax(scores, dim=-1)

    attention_mean = attention.mean(dim=1)

    normalized = F.normalize(
        attention_mean,
        dim=-1
    )

    similarity = torch.matmul(
        normalized,
        normalized.transpose(-1, -2)
    )

    mask = ~torch.eye(
        num_anchors,
        device=images.device,
        dtype=torch.bool
    )

    print(
        "Mean spatial-anchor attention-map similarity:",
        similarity[:, mask].mean().item()
    )


spatial_anchor_attention_similarity(
    rart_32,
    sample_images
)

Mean spatial-anchor attention-map similarity: 0.7570337653160095


In [ ]:
# Image
#   ↓
# Patch embedding
#   ↓
# 196 patches
#   ├──────────────→ local attention
#   │
#   └→ spatial anchor initialization
#              ↓
#        16 anchors
#              ↓
#       anchor self-attention
#              ↓
#       anchor → patches

In [71]:
block_16 = rart_16_spatial.blocks[0]

print(block_16)
print("\nPatch-to-anchor module:")
print(block_16.patch_to_anchor)

print("\nParameter shapes:")
for name, param in block_16.patch_to_anchor.named_parameters():
    print(f"{name}: {tuple(param.shape)}")

AnchorTransformerBlock(
  (patch_norm_local): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (patch_norm_global): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (anchor_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (ffn_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (local_attention): LocalWindowAttention(
    (qkv): Linear(in_features=384, out_features=1152, bias=True)
    (projection): Linear(in_features=384, out_features=384, bias=True)
    (attention_dropout): Dropout(p=0.0, inplace=False)
    (projection_dropout): Dropout(p=0.0, inplace=False)
  )
  (patch_to_anchor): PatchToAnchorAttention(
    (q_projection): Linear(in_features=384, out_features=384, bias=True)
    (kv_projection): Linear(in_features=384, out_features=768, bias=True)
    (output_projection): Linear(in_features=384, out_features=384, bias=True)
    (attention_dropout): Dropout(p=0.0, inplace=False)
    (projection_dropout): Dropout(p=0.0, inplace=False)
  )
  (an

In [72]:
@torch.no_grad()
def fullwidth_anchor_attention_similarity(model, images):
    model.eval()

    x = model.patch_embed(images)
    x = model.input_embedding(x)

    patches = x[:, 1:]

    block = model.blocks[0]
    module = block.patch_to_anchor

    patches_norm = block.patch_norm_global(patches)

    q = module.q_projection(
        model.anchor_tokens.expand(
            images.size(0),
            -1,
            -1
        )
    )

    kv = module.kv_projection(patches_norm)

    k, v = kv.chunk(2, dim=-1)

    B, A, D = q.shape
    N = k.shape[1]

    q = q.view(
        B, A,
        module.num_heads,
        D // module.num_heads
    ).transpose(1, 2)

    k = k.view(
        B, N,
        module.num_heads,
        D // module.num_heads
    ).transpose(1, 2)

    scores = torch.matmul(
        q,
        k.transpose(-2, -1)
    ) * module.scale

    attention = torch.softmax(scores, dim=-1)

    attention_mean = attention.mean(dim=1)

    normalized = F.normalize(
        attention_mean,
        dim=-1
    )

    similarity = torch.matmul(
        normalized,
        normalized.transpose(-1, -2)
    )

    mask = ~torch.eye(
        A,
        device=images.device,
        dtype=torch.bool
    )

    entropy = -(
        attention *
        torch.log(attention.clamp_min(1e-9))
    ).sum(dim=-1)

    effective_patches = entropy.exp().mean()

    print(
        f"Mean attention-map similarity: "
        f"{similarity[:, mask].mean().item():.4f}"
    )

    print(
        f"Effective patches attended: "
        f"{effective_patches.item():.2f} / {N}"
    )


fullwidth_anchor_attention_similarity(
    rart_16_spatial,
    sample_images
)

Mean attention-map similarity: 1.0000
Effective patches attended: 195.97 / 196


In [73]:
@torch.no_grad()
def fullwidth_spatial_attention_similarity(model, images):
    model.eval()

    x = model.patch_embed(images)
    x = model.input_embedding(x)

    patches = x[:, 1:]

    B, N, D = patches.shape
    grid_size = int(N ** 0.5)

    row_positions = torch.linspace(
        0,
        grid_size - 1,
        4,
        device=images.device
    ).round().long()

    col_positions = torch.linspace(
        0,
        grid_size - 1,
        4,
        device=images.device
    ).round().long()

    indices = torch.tensor(
        [
            r * grid_size + c
            for r in row_positions
            for c in col_positions
        ],
        device=images.device
    )

    spatial_anchors = patches[:, indices]

    block = model.blocks[0]
    module = block.patch_to_anchor

    patches_norm = block.patch_norm_global(patches)

    q = module.q_projection(
        spatial_anchors
    )

    kv = module.kv_projection(
        patches_norm
    )

    k, v = kv.chunk(2, dim=-1)

    q = q.view(
        B,
        16,
        module.num_heads,
        D // module.num_heads
    ).transpose(1, 2)

    k = k.view(
        B,
        N,
        module.num_heads,
        D // module.num_heads
    ).transpose(1, 2)

    scores = torch.matmul(
        q,
        k.transpose(-2, -1)
    ) * module.scale

    attention = torch.softmax(scores, dim=-1)

    attention_mean = attention.mean(dim=1)

    normalized = F.normalize(
        attention_mean,
        dim=-1
    )

    similarity = torch.matmul(
        normalized,
        normalized.transpose(-1, -2)
    )

    mask = ~torch.eye(
        16,
        device=images.device,
        dtype=torch.bool
    )

    entropy = -(
        attention *
        torch.log(attention.clamp_min(1e-9))
    ).sum(dim=-1)

    effective_patches = entropy.exp().mean()

    print(
        f"Mean attention-map similarity: "
        f"{similarity[:, mask].mean().item():.4f}"
    )

    print(
        f"Effective patches attended: "
        f"{effective_patches.item():.2f} / {N}"
    )


fullwidth_spatial_attention_similarity(
    rart_16_spatial,
    sample_images
)

Mean attention-map similarity: 0.9890
Effective patches attended: 192.15 / 196


In [74]:
@torch.no_grad()
def spatial_region_similarity(model, images, window_size=3):
    model.eval()

    x = model.patch_embed(images)
    x = model.input_embedding(x)

    patches = x[:, 1:]

    B, N, D = patches.shape
    grid_size = int(N ** 0.5)

    feature_grid = patches.reshape(
        B,
        grid_size,
        grid_size,
        D
    )

    centers = torch.linspace(
        0,
        grid_size - 1,
        4,
        device=images.device
    ).round().long()

    regions = []

    radius = window_size // 2

    for r in centers:
        for c in centers:
            r0 = max(0, r.item() - radius)
            r1 = min(grid_size, r.item() + radius + 1)
            c0 = max(0, c.item() - radius)
            c1 = min(grid_size, c.item() + radius + 1)

            region = feature_grid[
                :,
                r0:r1,
                c0:c1
            ].mean(dim=(1, 2))

            regions.append(region)

    regions = torch.stack(regions, dim=1)

    regions = F.normalize(
        regions,
        dim=-1
    )

    similarity = torch.matmul(
        regions,
        regions.transpose(-1, -2)
    )

    mask = ~torch.eye(
        16,
        device=images.device,
        dtype=torch.bool
    )

    print(
        f"Mean regional feature similarity: "
        f"{similarity[:, mask].mean().item():.4f}"
    )

spatial_region_similarity(
    rart_16_spatial,
    sample_images,
    window_size=3
)

Mean regional feature similarity: 0.3466


In [75]:
@torch.no_grad()
def local_anchor_routing_diagnostic(model, images, window_size=5):
    model.eval()

    x = model.patch_embed(images)
    x = model.input_embedding(x)

    patches = x[:, 1:]

    B, N, D = patches.shape
    grid_size = int(N ** 0.5)

    block = model.blocks[0]
    module = block.patch_to_anchor

    patches_norm = block.patch_norm_global(patches)

    q = module.q_projection(
        model.anchor_tokens.expand(B, -1, -1)
    )

    kv = module.kv_projection(patches_norm)
    k, v = kv.chunk(2, dim=-1)

    A = q.shape[1]
    H = module.num_heads
    head_dim = D // H

    q = q.view(B, A, H, head_dim).transpose(1, 2)
    k = k.view(B, N, H, head_dim).transpose(1, 2)

    scores = torch.matmul(
        q,
        k.transpose(-2, -1)
    ) * module.scale

    radius = window_size // 2

    centers = torch.linspace(
        0,
        grid_size - 1,
        4,
        device=images.device
    ).round().long()

    mask = torch.zeros(
        A,
        N,
        device=images.device,
        dtype=torch.bool
    )

    anchor_idx = 0

    for r in centers:
        for c in centers:
            for rr in range(grid_size):
                for cc in range(grid_size):
                    if (
                        abs(rr - r.item()) <= radius
                        and abs(cc - c.item()) <= radius
                    ):
                        patch_idx = rr * grid_size + cc
                        mask[anchor_idx, patch_idx] = True

            anchor_idx += 1

    scores = scores.masked_fill(
        ~mask[None, None],
        float("-inf")
    )

    attention = torch.softmax(scores, dim=-1)

    attention_mean = attention.mean(dim=1)

    normalized = F.normalize(
        attention_mean,
        dim=-1
    )

    similarity = torch.matmul(
        normalized,
        normalized.transpose(-1, -2)
    )

    similarity_mask = ~torch.eye(
        A,
        device=images.device,
        dtype=torch.bool
    )

    entropy = -(
        attention *
        torch.log(attention.clamp_min(1e-9))
    ).sum(dim=-1)

    effective_patches = entropy.exp().mean()

    print(
        f"Window size: {window_size}x{window_size}"
    )

    print(
        f"Mean attention-map similarity: "
        f"{similarity[:, similarity_mask].mean().item():.4f}"
    )

    print(
        f"Effective patches attended: "
        f"{effective_patches.item():.2f} / {N}"
    )

    print(
        f"Allowed patches per anchor: "
        f"{mask.sum(dim=-1).float().mean().item():.1f}"
    )


local_anchor_routing_diagnostic(
    rart_16_spatial,
    sample_images,
    window_size=5
)

Window size: 5x5
Mean attention-map similarity: 0.0389
Effective patches attended: 16.00 / 196
Allowed patches per anchor: 16.0


# Local → Global → Local block

In [82]:
class LocalGlobalLocalBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0,
        window_size=5
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = anchor_heads
        self.num_anchors = num_anchors
        self.window_size = window_size

        self.patch_norm_local = nn.LayerNorm(embed_dim)
        self.patch_norm_global = nn.LayerNorm(embed_dim)
        self.anchor_norm = nn.LayerNorm(anchor_dim)
        self.ffn_norm = nn.LayerNorm(embed_dim)

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=7
        )

        self.patch_to_anchor = CompressedPatchToAnchor(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_self_attention = CompressedAnchorSelfAttention(
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_to_patch = CompressedAnchorToPatch(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        hidden_dim = int(embed_dim * mlp_ratio)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

        self.alpha = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.ones(1))

    def _local_mask(self, grid_size, device):
        radius = self.window_size // 2

        side = int(self.num_anchors ** 0.5)

        centers = torch.linspace(
            0,
            grid_size - 1,
            side,
            device=device
        ).round().long()

        mask = torch.zeros(
            self.num_anchors,
            grid_size * grid_size,
            device=device,
            dtype=torch.bool
        )

        idx = 0

        for r in centers:
            for c in centers:

                for rr in range(grid_size):
                    for cc in range(grid_size):

                        if (
                            abs(rr - r.item()) <= radius
                            and abs(cc - c.item()) <= radius
                        ):
                            mask[
                                idx,
                                rr * grid_size + cc
                            ] = True

                idx += 1

        return mask

    def _patch_to_anchor_local(
        self,
        patches,
        anchors,
        mask
    ):
        B, N, _ = patches.shape
        A = anchors.shape[1]

        module = self.patch_to_anchor

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            anchors
        )

        kv = module.kv_projection(
            patch_proj
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            N,
            H,
            head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            N,
            H,
            head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) * module.scale

        scores = scores.masked_fill(
            ~mask[None, None],
            float("-inf")
        )

        attention = torch.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(
            attention,
            v
        )

        out = out.transpose(
            1,
            2
        ).reshape(
            B,
            A,
            D
        )

        return module.output_projection(
            out
        )

    def _anchor_to_patch_local(
        self,
        patches,
        anchors,
        mask
    ):
        B, N, _ = patches.shape
        A = anchors.shape[1]

        module = self.anchor_to_patch

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            patch_proj
        )

        kv = module.kv_projection(
            anchors
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            N,
            H,
            head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) * module.scale

        local_mask = mask.T

        scores = scores.masked_fill(
            ~local_mask[None, None],
            float("-inf")
        )

        attention = torch.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(
            attention,
            v
        )

        out = out.transpose(
            1,
            2
        ).reshape(
            B,
            N,
            D
        )

        return module.output_projection(
            out
        )

    def forward(
        self,
        patches,
        anchors
    ):
        B, N, D = patches.shape

        grid_size = int(N ** 0.5)

        patch_grid = patches.reshape(
            B,
            grid_size,
            grid_size,
            D
        )

        local = self.local_attention(
            self.patch_norm_local(
                patch_grid
            )
        )

        local = local.reshape(
            B,
            N,
            D
        )

        patches_local = patches + local

        patches_norm = self.patch_norm_global(
            patches_local
        )

        mask = self._local_mask(
            grid_size,
            patches.device
        )

        anchor_update = self._patch_to_anchor_local(
            patches_norm,
            anchors,
            mask
        )

        anchors = (
            anchors
            + self.alpha * anchor_update
        )

        anchors = (
            anchors
            + self.anchor_self_attention(
                self.anchor_norm(anchors)
            )
        )

        patch_update = self._anchor_to_patch_local(
            patches_norm,
            anchors,
            mask
        )

        patches = (
            patches_local
            + self.beta * patch_update
        )

        patches = (
            patches
            + self.ffn(
                self.ffn_norm(patches)
            )
        )

        return patches, anchors

# LGL RART

In [83]:
lgl_rart = LocalGlobalLocalRART(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    mlp_ratio=3.0,
    window_size=5
).to(device)

num_params = sum(
    p.numel()
    for p in lgl_rart.parameters()
)

print(f"Parameters: {num_params:,}")

with torch.no_grad():
    output = lgl_rart(sample_images.to(device))

print(f"Output shape: {output.shape}")

Parameters: 11,272,176
Output shape: torch.Size([128, 100])


# BenchMark LGL RART

In [84]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

benchmark_input = sample_images.to(device)

avg_time, peak_memory = benchmark_model(
    lgl_rart,
    benchmark_input,
    warmup=10,
    runs=50
)

throughput = benchmark_input.size(0) / avg_time

print(
    f"Average inference time: "
    f"{avg_time * 1000:.3f} ms"
)

print(
    f"Peak GPU memory: "
    f"{peak_memory / 1024**2:.2f} MB"
)

print(
    f"Throughput: "
    f"{throughput:.2f} img/s"
)

Average inference time: 426.030 ms
Peak GPU memory: 3956.67 MB
Throughput: 300.45 img/s


In [85]:
class OptimizedLocalGlobalLocalBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0,
        window_size=5
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = anchor_heads
        self.num_anchors = num_anchors
        self.window_size = window_size

        self.patch_norm_local = nn.LayerNorm(embed_dim)
        self.patch_norm_global = nn.LayerNorm(embed_dim)
        self.anchor_norm = nn.LayerNorm(anchor_dim)
        self.ffn_norm = nn.LayerNorm(embed_dim)

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=7
        )

        self.patch_to_anchor = CompressedPatchToAnchor(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_self_attention = CompressedAnchorSelfAttention(
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_to_patch = CompressedAnchorToPatch(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        hidden_dim = int(embed_dim * mlp_ratio)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

        self.alpha = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.ones(1))

    def _build_indices(self, grid_size, device):
        radius = self.window_size // 2

        side = int(self.num_anchors ** 0.5)

        centers = torch.linspace(
            0,
            grid_size - 1,
            side,
            device=device
        ).round().long()

        indices = []

        for r in centers:
            for c in centers:
                local = []

                for rr in range(grid_size):
                    for cc in range(grid_size):

                        if (
                            abs(rr - r.item()) <= radius
                            and abs(cc - c.item()) <= radius
                        ):
                            local.append(
                                rr * grid_size + cc
                            )

                indices.append(local)

        max_len = max(len(x) for x in indices)

        padded = []

        for x in indices:
            if len(x) < max_len:
                x = x + [x[-1]] * (max_len - len(x))
            padded.append(x)

        return torch.tensor(
            padded,
            device=device,
            dtype=torch.long
        )

    def _patch_to_anchor_local(
        self,
        patches,
        anchors,
        indices
    ):
        B, N, _ = patches.shape
        A, K = indices.shape

        module = self.patch_to_anchor

        patch_proj = module.patch_projection(
            patches
        )

        local_patches = patch_proj[
            :,
            indices,
            :
        ]

        q = module.q_projection(
            anchors
        )

        kv = module.kv_projection(
            local_patches
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            A,
            K,
            H,
            head_dim
        ).permute(
            0,
            3,
            1,
            2,
            4
        )

        v = v.view(
            B,
            A,
            K,
            H,
            head_dim
        ).permute(
            0,
            3,
            1,
            2,
            4
        )

        scores = (
            q.unsqueeze(3) * k
        ).sum(dim=-1) * module.scale

        attention = torch.softmax(
            scores,
            dim=-1
        )

        out = (
            attention.unsqueeze(-1) * v
        ).sum(dim=-2)

        out = out.permute(
            0,
            2,
            1,
            3
        ).reshape(
            B,
            A,
            D
        )

        return module.output_projection(
            out
        )

    def _anchor_to_patch_local(
        self,
        patches,
        anchors,
        indices
    ):
        B, N, _ = patches.shape
        A, K = indices.shape

        module = self.anchor_to_patch

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            patch_proj
        )

        kv = module.kv_projection(
            anchors
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            N,
            H,
            head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        local_mask = torch.zeros(
            N,
            A,
            device=patches.device,
            dtype=torch.bool
        )

        local_mask[
            indices.reshape(-1),
            torch.arange(
                A,
                device=patches.device
            ).repeat_interleave(K)
        ] = True

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) * module.scale

        scores = scores.masked_fill(
            ~local_mask[None, None],
            float("-inf")
        )

        attention = torch.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(
            attention,
            v
        )

        out = out.transpose(
            1,
            2
        ).reshape(
            B,
            N,
            D
        )

        return module.output_projection(
            out
        )

    def forward(
        self,
        patches,
        anchors
    ):
        B, N, D = patches.shape

        grid_size = int(N ** 0.5)

        patch_grid = patches.reshape(
            B,
            grid_size,
            grid_size,
            D
        )

        local = self.local_attention(
            self.patch_norm_local(
                patch_grid
            )
        )

        local = local.reshape(
            B,
            N,
            D
        )

        patches_local = patches + local

        patches_norm = self.patch_norm_global(
            patches_local
        )

        indices = self._build_indices(
            grid_size,
            patches.device
        )

        anchor_update = self._patch_to_anchor_local(
            patches_norm,
            anchors,
            indices
        )

        anchors = (
            anchors
            + self.alpha * anchor_update
        )

        anchors = (
            anchors
            + self.anchor_self_attention(
                self.anchor_norm(anchors)
            )
        )

        patch_update = self._anchor_to_patch_local(
            patches_norm,
            anchors,
            indices
        )

        patches = (
            patches_local
            + self.beta * patch_update
        )

        patches = (
            patches
            + self.ffn(
                self.ffn_norm(patches)
            )
        )

        return patches, anchors

# Optimized Version

In [88]:
class OptimizedLocalGlobalLocalBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0,
        window_size=5
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = anchor_heads
        self.num_anchors = num_anchors
        self.window_size = window_size

        self.patch_norm_local = nn.LayerNorm(embed_dim)
        self.patch_norm_global = nn.LayerNorm(embed_dim)
        self.anchor_norm = nn.LayerNorm(anchor_dim)
        self.ffn_norm = nn.LayerNorm(embed_dim)

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=7
        )

        self.patch_to_anchor = CompressedPatchToAnchor(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_self_attention = CompressedAnchorSelfAttention(
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_to_patch = CompressedAnchorToPatch(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        hidden_dim = int(embed_dim * mlp_ratio)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

        self.alpha = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.ones(1))

    def _build_indices(self, grid_size, device):
        radius = self.window_size // 2

        side = int(self.num_anchors ** 0.5)

        centers = torch.linspace(
            0,
            grid_size - 1,
            side,
            device=device
        ).round().long()

        indices = []

        for r in centers:
            for c in centers:
                local = []

                for rr in range(grid_size):
                    for cc in range(grid_size):

                        if (
                            abs(rr - r.item()) <= radius
                            and abs(cc - c.item()) <= radius
                        ):
                            local.append(
                                rr * grid_size + cc
                            )

                indices.append(local)

        max_len = max(len(x) for x in indices)

        padded = []

        for x in indices:
            if len(x) < max_len:
                x = x + [x[-1]] * (max_len - len(x))
            padded.append(x)

        return torch.tensor(
            padded,
            device=device,
            dtype=torch.long
        )

    def _patch_to_anchor_local(
        self,
        patches,
        anchors,
        indices
    ):
        B, N, _ = patches.shape
        A, K = indices.shape

        module = self.patch_to_anchor

        patch_proj = module.patch_projection(
            patches
        )

        local_patches = patch_proj[
            :,
            indices,
            :
        ]

        q = module.q_projection(
            anchors
        )

        kv = module.kv_projection(
            local_patches
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            A,
            K,
            H,
            head_dim
        ).permute(
            0,
            3,
            1,
            2,
            4
        )

        v = v.view(
            B,
            A,
            K,
            H,
            head_dim
        ).permute(
            0,
            3,
            1,
            2,
            4
        )

        scores = (
            q.unsqueeze(3) * k
        ).sum(dim=-1) * module.scale

        attention = torch.softmax(
            scores,
            dim=-1
        )

        out = (
            attention.unsqueeze(-1) * v
        ).sum(dim=-2)

        out = out.permute(
            0,
            2,
            1,
            3
        ).reshape(
            B,
            A,
            D
        )

        return module.output_projection(
            out
        )

    def _anchor_to_patch_local(
        self,
        patches,
        anchors,
        indices
    ):
        B, N, _ = patches.shape
        A, K = indices.shape

        module = self.anchor_to_patch

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            patch_proj
        )

        kv = module.kv_projection(
            anchors
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            N,
            H,
            head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            A,
            H,
            head_dim
        ).transpose(1, 2)

        local_mask = torch.zeros(
            N,
            A,
            device=patches.device,
            dtype=torch.bool
        )

        local_mask[
            indices.reshape(-1),
            torch.arange(
                A,
                device=patches.device
            ).repeat_interleave(K)
        ] = True

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) * module.scale

        scores = scores.masked_fill(
            ~local_mask[None, None],
            float("-inf")
        )

        attention = torch.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(
            attention,
            v
        )

        out = out.transpose(
            1,
            2
        ).reshape(
            B,
            N,
            D
        )

        return module.output_projection(
            out
        )

    def forward(
        self,
        patches,
        anchors
    ):
        B, N, D = patches.shape

        grid_size = int(N ** 0.5)

        patch_grid = patches.reshape(
            B,
            grid_size,
            grid_size,
            D
        )

        local = self.local_attention(
            self.patch_norm_local(
                patch_grid
            )
        )

        local = local.reshape(
            B,
            N,
            D
        )

        patches_local = patches + local

        patches_norm = self.patch_norm_global(
            patches_local
        )

        indices = self._build_indices(
            grid_size,
            patches.device
        )

        anchor_update = self._patch_to_anchor_local(
            patches_norm,
            anchors,
            indices
        )

        anchors = (
            anchors
            + self.alpha * anchor_update
        )

        anchors = (
            anchors
            + self.anchor_self_attention(
                self.anchor_norm(anchors)
            )
        )

        patch_update = self._anchor_to_patch_local(
            patches_norm,
            anchors,
            indices
        )

        patches = (
            patches_local
            + self.beta * patch_update
        )

        patches = (
            patches
            + self.ffn(
                self.ffn_norm(patches)
            )
        )

        return patches, anchors

In [89]:
class OptimizedLocalGlobalLocalRART(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        anchor_dim=128,
        depth=6,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0,
        window_size=5
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            embed_dim=embed_dim,
            num_patches=(image_size // patch_size) ** 2
        )

        self.anchor_tokens = nn.Parameter(
            torch.randn(
                1,
                num_anchors,
                anchor_dim
            ) * 0.02
        )

        self.blocks = nn.ModuleList([
            OptimizedLocalGlobalLocalBlock(
                embed_dim=embed_dim,
                anchor_dim=anchor_dim,
                num_heads=num_heads,
                anchor_heads=anchor_heads,
                num_anchors=num_anchors,
                mlp_ratio=mlp_ratio,
                window_size=window_size
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(
            embed_dim
        )

        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):

        x = self.patch_embed(x)

        x = self.input_embedding(x)

        patches = x[:, 1:]

        anchors = self.anchor_tokens.expand(
            x.size(0),
            -1,
            -1
        )

        for block in self.blocks:

            patches, anchors = block(
                patches,
                anchors
            )

        patches = self.norm(
            patches
        )

        x = patches.mean(
            dim=1
        )

        return self.head(x)


lgl_rart_opt = OptimizedLocalGlobalLocalRART(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    mlp_ratio=3.0,
    window_size=5
).to(device)

num_params = sum(
    p.numel()
    for p in lgl_rart_opt.parameters()
)

print(
    f"Parameters: {num_params:,}"
)

with torch.no_grad():

    output = lgl_rart_opt(
        sample_images.to(device)
    )

print(
    f"Output shape: {output.shape}"
)

Parameters: 11,272,176
Output shape: torch.Size([128, 100])


In [90]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

benchmark_input = sample_images.to(device)

avg_time, peak_memory = benchmark_model(
    lgl_rart_opt,
    benchmark_input,
    warmup=10,
    runs=50
)

throughput = benchmark_input.size(0) / avg_time

print(
    f"Average inference time: "
    f"{avg_time * 1000:.3f} ms"
)

print(
    f"Peak GPU memory: "
    f"{peak_memory / 1024**2:.2f} MB"
)

print(
    f"Throughput: "
    f"{throughput:.2f} img/s"
)

Average inference time: 399.040 ms
Peak GPU memory: 4000.50 MB
Throughput: 320.77 img/s


In [91]:
import time
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    lgl_rart_opt.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

scaler = torch.amp.GradScaler("cuda")

for epoch in range(10):

    lgl_rart_opt.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    start = time.perf_counter()

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):
            outputs = lgl_rart_opt(images)
            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_loss += (
            loss.item() * images.size(0)
        )

        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        train_total += images.size(0)

    scheduler.step()

    train_loss /= train_total
    train_acc = (
        100.0 * train_correct / train_total
    )

    lgl_rart_opt.eval()

    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16
            ):
                outputs = lgl_rart_opt(images)
                loss = criterion(
                    outputs,
                    labels
                )

            test_loss += (
                loss.item() * images.size(0)
            )

            test_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()

            test_total += images.size(0)

    test_loss /= test_total
    test_acc = (
        100.0 * test_correct / test_total
    )

    elapsed = time.perf_counter() - start

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Test Loss {test_loss:.4f} | "
        f"Test Acc {test_acc:.2f}% | "
        f"{elapsed / 60:.2f}m"
    )

Epoch 01 | Train Loss 3.8368 | Train Acc 10.96% | Test Loss 3.4477 | Test Acc 17.18% | 3.84m
Epoch 02 | Train Loss 3.2196 | Train Acc 20.83% | Test Loss 3.0761 | Test Acc 24.04% | 3.84m
Epoch 03 | Train Loss 2.8846 | Train Acc 27.33% | Test Loss 2.8039 | Test Acc 29.55% | 3.85m
Epoch 04 | Train Loss 2.6061 | Train Acc 32.66% | Test Loss 2.6014 | Test Acc 34.09% | 3.88m
Epoch 05 | Train Loss 2.3740 | Train Acc 37.88% | Test Loss 2.4172 | Test Acc 37.39% | 3.86m
Epoch 06 | Train Loss 2.1557 | Train Acc 42.56% | Test Loss 2.2918 | Test Acc 39.48% | 3.85m
Epoch 07 | Train Loss 1.9534 | Train Acc 47.36% | Test Loss 2.1729 | Test Acc 42.70% | 3.84m
Epoch 08 | Train Loss 1.7635 | Train Acc 52.01% | Test Loss 2.0965 | Test Acc 44.18% | 3.86m
Epoch 09 | Train Loss 1.6113 | Train Acc 56.02% | Test Loss 2.0563 | Test Acc 44.96% | 3.85m
Epoch 10 | Train Loss 1.5246 | Train Acc 58.44% | Test Loss 2.0365 | Test Acc 45.38% | 3.85m


In [92]:
@torch.no_grad()
def inspect_lgl_anchors(model, images):

    model.eval()

    x = model.patch_embed(images)
    x = model.input_embedding(x)

    patches = x[:, 1:]

    anchors = model.anchor_tokens.expand(
        images.size(0),
        -1,
        -1
    )

    for i, block in enumerate(model.blocks):

        patches_grid = patches.reshape(
            images.size(0),
            14,
            14,
            -1
        )

        local = block.local_attention(
            block.patch_norm_local(
                patches_grid
            )
        )

        local = local.reshape(
            images.size(0),
            196,
            -1
        )

        patches_local = patches + local

        patches_norm = block.patch_norm_global(
            patches_local
        )

        indices = block._build_indices(
            14,
            images.device
        )

        anchor_update = block._patch_to_anchor_local(
            patches_norm,
            anchors,
            indices
        )

        anchors = (
            anchors
            + block.alpha * anchor_update
        )

        normalized = F.normalize(
            anchors,
            dim=-1
        )

        similarity = torch.matmul(
            normalized,
            normalized.transpose(-1, -2)
        )

        mask = ~torch.eye(
            16,
            device=images.device,
            dtype=torch.bool
        )

        print(
            f"Block {i + 1}: "
            f"Anchor similarity = "
            f"{similarity[:, mask].mean().item():.4f}"
        )

        anchors = (
            anchors
            + block.anchor_self_attention(
                block.anchor_norm(anchors)
            )
        )

        patches = (
            patches_local
            + block.beta * block._anchor_to_patch_local(
                patches_norm,
                anchors,
                indices
            )
        )

        patches = (
            patches
            + block.ffn(
                block.ffn_norm(patches)
            )
        )


inspect_lgl_anchors(
    lgl_rart_opt,
    sample_images.to(device)
)

Block 1: Anchor similarity = 0.2619
Block 2: Anchor similarity = 0.2550
Block 3: Anchor similarity = 0.2976
Block 4: Anchor similarity = 0.3295
Block 5: Anchor similarity = 0.3682
Block 6: Anchor similarity = 0.4227


In [93]:
checkpoint_path = "/kaggle/working/lgl_rart_16_epoch10_checkpoint.pth"

torch.save(
    {
        "model_state_dict": lgl_rart_opt.state_dict(),
        "epoch": 10,
        "test_accuracy": 45.38,
        "train_accuracy": 58.44,
        "params": 11272176,
        "config": {
            "image_size": 224,
            "patch_size": 16,
            "num_classes": 100,
            "embed_dim": 384,
            "anchor_dim": 128,
            "depth": 6,
            "num_heads": 6,
            "anchor_heads": 4,
            "num_anchors": 16,
            "mlp_ratio": 3.0,
            "window_size": 5
        }
    },
    checkpoint_path
)

print(f"Checkpoint saved to: {checkpoint_path}")

Checkpoint saved to: /kaggle/working/lgl_rart_16_epoch10_checkpoint.pth


In [94]:
def build_partition_indices(
    grid_size=14,
    num_anchors=16,
    device="cuda"
):
    side = int(num_anchors ** 0.5)

    anchor_rows = torch.linspace(
        0,
        grid_size - 1,
        side,
        device=device
    )

    anchor_cols = torch.linspace(
        0,
        grid_size - 1,
        side,
        device=device
    )

    anchor_centers = torch.stack(
        torch.meshgrid(
            anchor_rows,
            anchor_cols,
            indexing="ij"
        ),
        dim=-1
    ).reshape(
        num_anchors,
        2
    )

    patch_coords = torch.stack(
        torch.meshgrid(
            torch.arange(
                grid_size,
                device=device
            ),
            torch.arange(
                grid_size,
                device=device
            ),
            indexing="ij"
        ),
        dim=-1
    ).reshape(
        grid_size * grid_size,
        2
    ).float()

    distances = torch.cdist(
        patch_coords,
        anchor_centers
    )

    assignments = distances.argmin(
        dim=-1
    )

    counts = torch.bincount(
        assignments,
        minlength=num_anchors
    )

    print(
        f"Number of regions: {num_anchors}"
    )

    print(
        f"Min patches/region: {counts.min().item()}"
    )

    print(
        f"Max patches/region: {counts.max().item()}"
    )

    print(
        f"Mean patches/region: {counts.float().mean().item():.2f}"
    )

    return assignments


partition_indices = build_partition_indices(
    grid_size=14,
    num_anchors=16,
    device=device
)

print(partition_indices.shape)

Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
torch.Size([196])


# balanced 16-region partition

In [95]:
def build_balanced_partition_indices(
    grid_size=14,
    num_anchors=16,
    device="cuda"
):
    side = int(num_anchors ** 0.5)

    row_bins = torch.tensor(
        [0, 3, 7, 10, 14],
        device=device
    )

    col_bins = torch.tensor(
        [0, 3, 7, 10, 14],
        device=device
    )

    assignments = torch.empty(
        grid_size * grid_size,
        device=device,
        dtype=torch.long
    )

    for r in range(grid_size):
        for c in range(grid_size):

            row_region = min(
                (r * side) // grid_size,
                side - 1
            )

            col_region = min(
                (c * side) // grid_size,
                side - 1
            )

            anchor_idx = (
                row_region * side
                + col_region
            )

            patch_idx = (
                r * grid_size + c
            )

            assignments[
                patch_idx
            ] = anchor_idx

    counts = torch.bincount(
        assignments,
        minlength=num_anchors
    )

    print(
        f"Number of regions: {num_anchors}"
    )

    print(
        f"Min patches/region: "
        f"{counts.min().item()}"
    )

    print(
        f"Max patches/region: "
        f"{counts.max().item()}"
    )

    print(
        f"Mean patches/region: "
        f"{counts.float().mean().item():.2f}"
    )

    print(
        f"Region sizes: "
        f"{counts.tolist()}"
    )

    return assignments


balanced_partition_indices = (
    build_balanced_partition_indices(
        grid_size=14,
        num_anchors=16,
        device=device
    )
)


Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]


# partition routing block

In [96]:
class PartitionLocalGlobalLocalBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = anchor_heads
        self.num_anchors = num_anchors

        self.patch_norm_local = nn.LayerNorm(
            embed_dim
        )

        self.patch_norm_global = nn.LayerNorm(
            embed_dim
        )

        self.anchor_norm = nn.LayerNorm(
            anchor_dim
        )

        self.ffn_norm = nn.LayerNorm(
            embed_dim
        )

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=7
        )

        self.patch_to_anchor = CompressedPatchToAnchor(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_self_attention = (
            CompressedAnchorSelfAttention(
                anchor_dim=anchor_dim,
                num_heads=anchor_heads
            )
        )

        self.anchor_to_patch = CompressedAnchorToPatch(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        hidden_dim = int(
            embed_dim * mlp_ratio
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                embed_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                embed_dim
            )
        )

        self.alpha = nn.Parameter(
            torch.ones(1)
        )

        self.beta = nn.Parameter(
            torch.ones(1)
        )

    def _get_regions(
        self,
        grid_size,
        device
    ):
        return build_balanced_partition_indices(
            grid_size=grid_size,
            num_anchors=self.num_anchors,
            device=device
        )

    def _patch_to_anchor(
        self,
        patches,
        anchors,
        assignments
    ):
        B, N, _ = patches.shape
        A = self.num_anchors

        module = self.patch_to_anchor

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            anchors
        )

        kv = module.kv_projection(
            patch_proj
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        output = torch.zeros(
            B,
            A,
            D,
            device=patches.device,
            dtype=patches.dtype
        )

        for anchor_idx in range(A):

            patch_indices = torch.where(
                assignments == anchor_idx
            )[0]

            local_k = k[
                :,
                patch_indices
            ]

            local_v = v[
                :,
                patch_indices
            ]

            local_k = local_k.view(
                B,
                -1,
                H,
                head_dim
            ).permute(
                0,
                2,
                1,
                3
            )

            local_v = local_v.view(
                B,
                -1,
                H,
                head_dim
            ).permute(
                0,
                2,
                1,
                3
            )

            local_q = q[
                :,
                :,
                anchor_idx:anchor_idx + 1
            ]

            scores = torch.matmul(
                local_q,
                local_k.transpose(
                    -2,
                    -1
                )
            ) * module.scale

            attention = torch.softmax(
                scores,
                dim=-1
            )

            local_output = torch.matmul(
                attention,
                local_v
            )

            local_output = (
                local_output
                .squeeze(2)
                .transpose(1, 2)
                .reshape(B, D)
            )

            output[
                :,
                anchor_idx
            ] = local_output

        return module.output_projection(
            output
        )

    def _anchor_to_patch(
        self,
        patches,
        anchors,
        assignments
    ):
        B, N, _ = patches.shape
        A = self.num_anchors

        module = self.anchor_to_patch

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            patch_proj
        )

        kv = module.kv_projection(
            anchors
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            N,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        k = k.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        v = v.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        anchor_indices = (
            assignments
        )

        selected_k = k[
            :,
            :,
            anchor_indices
        ]

        selected_v = v[
            :,
            :,
            anchor_indices
        ]

        selected_k = selected_k.permute(
            0,
            1,
            3,
            2
        )

        selected_v = selected_v.permute(
            0,
            1,
            3,
            2
        )

        selected_k = selected_k.unsqueeze(
            2
        )

        selected_v = selected_v.unsqueeze(
            2
        )

        q = q.unsqueeze(3)

        scores = (
            q * selected_k
        ).sum(
            dim=-1
        ) * module.scale

        attention = torch.sigmoid(
            scores
        )

        output = (
            attention.unsqueeze(-1)
            * selected_v
        ).squeeze(3)

        output = output.permute(
            0,
            2,
            1,
            3
        ).reshape(
            B,
            N,
            D
        )

        return module.output_projection(
            output
        )

    def forward(
        self,
        patches,
        anchors
    ):
        B, N, D = patches.shape

        grid_size = int(
            N ** 0.5
        )

        patch_grid = patches.reshape(
            B,
            grid_size,
            grid_size,
            D
        )

        local = self.local_attention(
            self.patch_norm_local(
                patch_grid
            )
        )

        local = local.reshape(
            B,
            N,
            D
        )

        patches_local = (
            patches + local
        )

        patches_norm = (
            self.patch_norm_global(
                patches_local
            )
        )

        assignments = self._get_regions(
            grid_size,
            patches.device
        )

        anchor_update = self._patch_to_anchor(
            patches_norm,
            anchors,
            assignments
        )

        anchors = (
            anchors
            + self.alpha * anchor_update
        )

        anchors = (
            anchors
            + self.anchor_self_attention(
                self.anchor_norm(
                    anchors
                )
            )
        )

        patch_update = self._anchor_to_patch(
            patches_norm,
            anchors,
            assignments
        )

        patches = (
            patches_local
            + self.beta * patch_update
        )

        patches = (
            patches
            + self.ffn(
                self.ffn_norm(
                    patches
                )
            )
        )

        return patches, anchors

In [101]:
class PartitionLocalGlobalLocalBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = anchor_heads
        self.num_anchors = num_anchors

        self.patch_norm_local = nn.LayerNorm(
            embed_dim
        )

        self.patch_norm_global = nn.LayerNorm(
            embed_dim
        )

        self.anchor_norm = nn.LayerNorm(
            anchor_dim
        )

        self.ffn_norm = nn.LayerNorm(
            embed_dim
        )

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=7
        )

        self.patch_to_anchor = CompressedPatchToAnchor(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_self_attention = (
            CompressedAnchorSelfAttention(
                anchor_dim=anchor_dim,
                num_heads=anchor_heads
            )
        )

        self.anchor_to_patch = CompressedAnchorToPatch(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        hidden_dim = int(
            embed_dim * mlp_ratio
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                embed_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                embed_dim
            )
        )

        self.alpha = nn.Parameter(
            torch.ones(1)
        )

        self.beta = nn.Parameter(
            torch.ones(1)
        )

    def _get_regions(
        self,
        grid_size,
        device
    ):
        return build_balanced_partition_indices(
            grid_size=grid_size,
            num_anchors=self.num_anchors,
            device=device
        )

    def _patch_to_anchor(
        self,
        patches,
        anchors,
        assignments
    ):
        B, N, _ = patches.shape
        A = self.num_anchors

        module = self.patch_to_anchor

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            anchors
        )

        kv = module.kv_projection(
            patch_proj
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        k = k.view(
            B,
            N,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        v = v.view(
            B,
            N,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        outputs = []

        for anchor_idx in range(A):

            patch_indices = torch.where(
                assignments == anchor_idx
            )[0]

            local_k = k[
                :,
                :,
                patch_indices,
                :
            ]

            local_v = v[
                :,
                :,
                patch_indices,
                :
            ]

            local_q = q[
                :,
                :,
                anchor_idx:anchor_idx + 1,
                :
            ]

            scores = torch.matmul(
                local_q,
                local_k.transpose(
                    -2,
                    -1
                )
            ) * module.scale

            attention = torch.softmax(
                scores,
                dim=-1
            )

            local_output = torch.matmul(
                attention,
                local_v
            )

            local_output = (
                local_output
                .squeeze(2)
                .transpose(1, 2)
                .reshape(
                    B,
                    D
                )
            )

            outputs.append(
                local_output
            )

        output = torch.stack(
            outputs,
            dim=1
        )

        return module.output_projection(
            output
        )

    def _anchor_to_patch(
        self,
        patches,
        anchors,
        assignments
    ):
        B, N, _ = patches.shape
        A = self.num_anchors

        module = self.anchor_to_patch

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            patch_proj
        )

        kv = module.kv_projection(
            anchors
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            N,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        k = k.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        v = v.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        selected_k = k[
            :,
            :,
            assignments,
            :
        ]

        selected_v = v[
            :,
            :,
            assignments,
            :
        ]

        scores = (
            q * selected_k
        ).sum(
            dim=-1
        ) * module.scale

        attention = torch.sigmoid(
            scores
        )

        output = (
            attention.unsqueeze(-1)
            * selected_v
        )

        output = output.permute(
            0,
            2,
            1,
            3
        ).reshape(
            B,
            N,
            D
        )

        return module.output_projection(
            output
        )

    def forward(
        self,
        patches,
        anchors
    ):
        B, N, D = patches.shape

        grid_size = int(
            N ** 0.5
        )

        patch_grid = patches.reshape(
            B,
            grid_size,
            grid_size,
            D
        )

        local = self.local_attention(
            self.patch_norm_local(
                patch_grid
            )
        )

        local = local.reshape(
            B,
            N,
            D
        )

        patches_local = (
            patches + local
        )

        patches_norm = self.patch_norm_global(
            patches_local
        )

        assignments = self._get_regions(
            grid_size,
            patches.device
        )

        anchor_update = self._patch_to_anchor(
            patches_norm,
            anchors,
            assignments
        )

        anchors = (
            anchors
            + self.alpha * anchor_update
        )

        anchors = (
            anchors
            + self.anchor_self_attention(
                self.anchor_norm(
                    anchors
                )
            )
        )

        patch_update = self._anchor_to_patch(
            patches_norm,
            anchors,
            assignments
        )

        patches = (
            patches_local
            + self.beta * patch_update
        )

        patches = (
            patches
            + self.ffn(
                self.ffn_norm(
                    patches
                )
            )
        )

        return patches, anchors

In [102]:
class PartitionLocalGlobalLocalRART(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        anchor_dim=128,
        depth=6,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            embed_dim=embed_dim,
            num_patches=(
                image_size // patch_size
            ) ** 2
        )

        self.anchor_tokens = nn.Parameter(
            torch.randn(
                1,
                num_anchors,
                anchor_dim
            ) * 0.02
        )

        self.blocks = nn.ModuleList([
            PartitionLocalGlobalLocalBlock(
                embed_dim=embed_dim,
                anchor_dim=anchor_dim,
                num_heads=num_heads,
                anchor_heads=anchor_heads,
                num_anchors=num_anchors,
                mlp_ratio=mlp_ratio
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(
            embed_dim
        )

        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):

        x = self.patch_embed(x)

        x = self.input_embedding(x)

        patches = x[:, 1:]

        anchors = self.anchor_tokens.expand(
            x.size(0),
            -1,
            -1
        )

        for block in self.blocks:

            patches, anchors = block(
                patches,
                anchors
            )

        patches = self.norm(
            patches
        )

        x = patches.mean(
            dim=1
        )

        return self.head(x)


partition_rart = PartitionLocalGlobalLocalRART(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    mlp_ratio=3.0
).to(device)

num_params = sum(
    p.numel()
    for p in partition_rart.parameters()
)

print(
    f"Parameters: {num_params:,}"
)

with torch.no_grad():

    output = partition_rart(
        sample_images.to(device)
    )

print(
    f"Output shape: {output.shape}"
)

Parameters: 11,272,176
Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 

In [103]:
partition_indices = build_balanced_partition_indices(
    grid_size=14,
    num_anchors=16,
    device=device
)

print("Partition shape:", partition_indices.shape)
print("Unique regions:", partition_indices.unique().numel())
print(
    "All patches assigned:",
    partition_indices.numel() == 196
)

Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Partition shape: torch.Size([196])
Unique regions: 16
All patches assigned: True


In [104]:
class PartitionLocalGlobalLocalBlock(nn.Module):
    def __init__(
        self,
        embed_dim=384,
        anchor_dim=128,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.anchor_dim = anchor_dim
        self.num_heads = anchor_heads
        self.num_anchors = num_anchors

        self.patch_norm_local = nn.LayerNorm(
            embed_dim
        )

        self.patch_norm_global = nn.LayerNorm(
            embed_dim
        )

        self.anchor_norm = nn.LayerNorm(
            anchor_dim
        )

        self.ffn_norm = nn.LayerNorm(
            embed_dim
        )

        self.local_attention = LocalWindowAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            window_size=7
        )

        self.patch_to_anchor = CompressedPatchToAnchor(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        self.anchor_self_attention = (
            CompressedAnchorSelfAttention(
                anchor_dim=anchor_dim,
                num_heads=anchor_heads
            )
        )

        self.anchor_to_patch = CompressedAnchorToPatch(
            embed_dim=embed_dim,
            anchor_dim=anchor_dim,
            num_heads=anchor_heads
        )

        hidden_dim = int(
            embed_dim * mlp_ratio
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                embed_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                embed_dim
            )
        )

        self.alpha = nn.Parameter(
            torch.ones(1)
        )

        self.beta = nn.Parameter(
            torch.ones(1)
        )

    def _patch_to_anchor(
        self,
        patches,
        anchors,
        assignments
    ):
        B, N, _ = patches.shape
        A = self.num_anchors

        module = self.patch_to_anchor

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            anchors
        )

        kv = module.kv_projection(
            patch_proj
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        k = k.view(
            B,
            N,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        v = v.view(
            B,
            N,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        outputs = []

        for anchor_idx in range(A):

            patch_indices = torch.where(
                assignments == anchor_idx
            )[0]

            local_k = k[
                :,
                :,
                patch_indices,
                :
            ]

            local_v = v[
                :,
                :,
                patch_indices,
                :
            ]

            local_q = q[
                :,
                :,
                anchor_idx:anchor_idx + 1,
                :
            ]

            scores = torch.matmul(
                local_q,
                local_k.transpose(
                    -2,
                    -1
                )
            ) * module.scale

            attention = torch.softmax(
                scores,
                dim=-1
            )

            local_output = torch.matmul(
                attention,
                local_v
            )

            local_output = (
                local_output
                .squeeze(2)
                .transpose(1, 2)
                .reshape(
                    B,
                    D
                )
            )

            outputs.append(
                local_output
            )

        output = torch.stack(
            outputs,
            dim=1
        )

        return module.output_projection(
            output
        )

    def _anchor_to_patch(
        self,
        patches,
        anchors,
        assignments
    ):
        B, N, _ = patches.shape
        A = self.num_anchors

        module = self.anchor_to_patch

        patch_proj = module.patch_projection(
            patches
        )

        q = module.q_projection(
            patch_proj
        )

        kv = module.kv_projection(
            anchors
        )

        k, v = kv.chunk(
            2,
            dim=-1
        )

        D = q.shape[-1]
        H = module.num_heads
        head_dim = D // H

        q = q.view(
            B,
            N,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        k = k.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        v = v.view(
            B,
            A,
            H,
            head_dim
        ).permute(
            0,
            2,
            1,
            3
        )

        selected_k = k[
            :,
            :,
            assignments,
            :
        ]

        selected_v = v[
            :,
            :,
            assignments,
            :
        ]

        scores = (
            q * selected_k
        ).sum(
            dim=-1
        ) * module.scale

        attention = torch.sigmoid(
            scores
        )

        output = (
            attention.unsqueeze(-1)
            * selected_v
        )

        output = output.permute(
            0,
            2,
            1,
            3
        ).reshape(
            B,
            N,
            D
        )

        return module.output_projection(
            output
        )

    def forward(
        self,
        patches,
        anchors,
        assignments
    ):
        B, N, D = patches.shape

        grid_size = int(
            N ** 0.5
        )

        patch_grid = patches.reshape(
            B,
            grid_size,
            grid_size,
            D
        )

        local = self.local_attention(
            self.patch_norm_local(
                patch_grid
            )
        )

        local = local.reshape(
            B,
            N,
            D
        )

        patches_local = (
            patches + local
        )

        patches_norm = self.patch_norm_global(
            patches_local
        )

        anchor_update = self._patch_to_anchor(
            patches_norm,
            anchors,
            assignments
        )

        anchors = (
            anchors
            + self.alpha * anchor_update
        )

        anchors = (
            anchors
            + self.anchor_self_attention(
                self.anchor_norm(
                    anchors
                )
            )
        )

        patch_update = self._anchor_to_patch(
            patches_norm,
            anchors,
            assignments
        )

        patches = (
            patches_local
            + self.beta * patch_update
        )

        patches = (
            patches
            + self.ffn(
                self.ffn_norm(
                    patches
                )
            )
        )

        return patches, anchors

In [105]:
class PartitionLocalGlobalLocalRART(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=100,
        embed_dim=384,
        anchor_dim=128,
        depth=6,
        num_heads=6,
        anchor_heads=4,
        num_anchors=16,
        mlp_ratio=3.0
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        self.input_embedding = ViTInputEmbedding(
            embed_dim=embed_dim,
            num_patches=(
                image_size // patch_size
            ) ** 2
        )

        self.anchor_tokens = nn.Parameter(
            torch.randn(
                1,
                num_anchors,
                anchor_dim
            ) * 0.02
        )

        self.blocks = nn.ModuleList([
            PartitionLocalGlobalLocalBlock(
                embed_dim=embed_dim,
                anchor_dim=anchor_dim,
                num_heads=num_heads,
                anchor_heads=anchor_heads,
                num_anchors=num_anchors,
                mlp_ratio=mlp_ratio
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(
            embed_dim
        )

        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

        assignments = build_balanced_partition_indices(
            grid_size=image_size // patch_size,
            num_anchors=num_anchors,
            device="cpu"
        )

        self.register_buffer(
            "assignments",
            assignments,
            persistent=False
        )

    def forward(self, x):

        x = self.patch_embed(x)

        x = self.input_embedding(x)

        patches = x[:, 1:]

        anchors = self.anchor_tokens.expand(
            x.size(0),
            -1,
            -1
        )

        assignments = self.assignments.to(
            x.device
        )

        for block in self.blocks:

            patches, anchors = block(
                patches,
                anchors,
                assignments
            )

        patches = self.norm(
            patches
        )

        x = patches.mean(
            dim=1
        )

        return self.head(x)

In [106]:
partition_rart = PartitionLocalGlobalLocalRART(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    mlp_ratio=3.0
).to(device)

num_params = sum(
    p.numel()
    for p in partition_rart.parameters()
)

print(f"Parameters: {num_params:,}")

with torch.no_grad():
    output = partition_rart(
        sample_images.to(device)
    )

print(f"Output shape: {output.shape}")

Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Parameters: 11,272,176
Output shape: torch.Size([128, 100])


In [107]:
import time
import torch

partition_rart.eval()

for _ in range(10):
    with torch.no_grad():
        _ = partition_rart(
            sample_images.to(device)
        )

torch.cuda.synchronize()

start = time.perf_counter()

for _ in range(50):
    with torch.no_grad():
        _ = partition_rart(
            sample_images.to(device)
        )

torch.cuda.synchronize()

elapsed = time.perf_counter() - start

avg_time = elapsed / 50
throughput = sample_images.size(0) / avg_time

peak_memory = (
    torch.cuda.max_memory_allocated(device)
    / 1024**2
)

print(f"Average inference time: {avg_time * 1000:.3f} ms")
print(f"Peak GPU memory: {peak_memory:.2f} MB")
print(f"Throughput: {throughput:.2f} images/sec")

Average inference time: 219.152 ms
Peak GPU memory: 6686.93 MB
Throughput: 584.07 images/sec


In [108]:
partition_rart.eval()

patches = partition_rart.patch_embed(
    sample_images.to(device)
)

patches = partition_rart.input_embedding(
    patches
)

patches = patches[:, 1:]

anchors = partition_rart.anchor_tokens.expand(
    patches.size(0),
    -1,
    -1
)

block = partition_rart.blocks[0]

assignments = partition_rart.assignments

patches_norm = block.patch_norm_global(
    patches
)

torch.cuda.synchronize()

start = time.perf_counter()

with torch.no_grad():
    anchor_update = block._patch_to_anchor(
        patches_norm,
        anchors,
        assignments
    )

torch.cuda.synchronize()

patch_to_anchor_time = time.perf_counter() - start

anchors_updated = (
    anchors
    + block.alpha * anchor_update
)

torch.cuda.synchronize()

start = time.perf_counter()

with torch.no_grad():
    patch_update = block._anchor_to_patch(
        patches_norm,
        anchors_updated,
        assignments
    )

torch.cuda.synchronize()

anchor_to_patch_time = time.perf_counter() - start

print(
    f"Patch → Anchor: "
    f"{patch_to_anchor_time * 1000:.3f} ms"
)

print(
    f"Anchor → Patch: "
    f"{anchor_to_patch_time * 1000:.3f} ms"
)

print(
    f"Total routing: "
    f"{(patch_to_anchor_time + anchor_to_patch_time) * 1000:.3f} ms"
)

Patch → Anchor: 8.186 ms
Anchor → Patch: 3.971 ms
Total routing: 12.157 ms


In [109]:
import time
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

partition_rart.eval()

x = sample_images.to(device)

for _ in range(10):
    with torch.no_grad():
        _ = partition_rart(x)

torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats(device)

start = time.perf_counter()

for _ in range(50):
    with torch.no_grad():
        _ = partition_rart(x)

torch.cuda.synchronize()

elapsed = time.perf_counter() - start

avg_time = elapsed / 50
throughput = x.size(0) / avg_time

peak_memory = (
    torch.cuda.max_memory_allocated(device)
    / 1024**2
)

print(f"Average inference time: {avg_time * 1000:.3f} ms")
print(f"Peak GPU memory: {peak_memory:.2f} MB")
print(f"Throughput: {throughput:.2f} images/sec")

Average inference time: 221.430 ms
Peak GPU memory: 3982.33 MB
Throughput: 578.06 images/sec


In [111]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

print("Seed reset to 42")

Seed reset to 42


In [112]:
set_seed(42)

partition_rart = PartitionLocalGlobalLocalRART(
    image_size=224,
    patch_size=16,
    num_classes=100,
    embed_dim=384,
    anchor_dim=128,
    depth=6,
    num_heads=6,
    anchor_heads=4,
    num_anchors=16,
    mlp_ratio=3.0
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    partition_rart.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scaler = torch.amp.GradScaler(
    "cuda"
)

num_epochs = 10

for epoch in range(num_epochs):

    partition_rart.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    start_time = time.time()

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):
            outputs = partition_rart(
                images
            )

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        train_loss += (
            loss.item()
            * images.size(0)
        )

        train_correct += (
            outputs.argmax(
                dim=1
            ) == labels
        ).sum().item()

        train_total += images.size(0)

    train_loss /= train_total

    train_accuracy = (
        100.0
        * train_correct
        / train_total
    )

    partition_rart.eval()

    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16
            ):
                outputs = partition_rart(
                    images
                )

                loss = criterion(
                    outputs,
                    labels
                )

            test_loss += (
                loss.item()
                * images.size(0)
            )

            test_correct += (
                outputs.argmax(
                    dim=1
                ) == labels
            ).sum().item()

            test_total += images.size(0)

    test_loss /= test_total

    test_accuracy = (
        100.0
        * test_correct
        / test_total
    )

    elapsed = (
        time.time()
        - start_time
    ) / 60

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_accuracy:.2f}% | "
        f"{elapsed:.2f}m"
    )

Number of regions: 16
Min patches/region: 9
Max patches/region: 16
Mean patches/region: 12.25
Region sizes: [16, 12, 16, 12, 12, 9, 12, 9, 16, 12, 16, 12, 12, 9, 12, 9]
Epoch 01 | Train Loss: 3.8717 | Train Acc: 10.40% | Test Loss: 3.4842 | Test Acc: 16.60% | 2.32m
Epoch 02 | Train Loss: 3.2570 | Train Acc: 20.41% | Test Loss: 3.0773 | Test Acc: 24.19% | 2.32m
Epoch 03 | Train Loss: 2.9127 | Train Acc: 26.54% | Test Loss: 2.8169 | Test Acc: 28.90% | 2.31m
Epoch 04 | Train Loss: 2.6642 | Train Acc: 31.60% | Test Loss: 2.6309 | Test Acc: 32.55% | 2.30m
Epoch 05 | Train Loss: 2.4569 | Train Acc: 35.96% | Test Loss: 2.4912 | Test Acc: 35.73% | 2.30m
Epoch 06 | Train Loss: 2.2813 | Train Acc: 39.90% | Test Loss: 2.3791 | Test Acc: 38.35% | 2.31m
Epoch 07 | Train Loss: 2.1195 | Train Acc: 43.35% | Test Loss: 2.3115 | Test Acc: 40.08% | 2.32m
Epoch 08 | Train Loss: 1.9675 | Train Acc: 46.76% | Test Loss: 2.2150 | Test Acc: 42.19% | 2.30m
Epoch 09 | Train Loss: 1.8247 | Train Acc: 50.03% | Tes

In [113]:
checkpoint_path = "/kaggle/working/partition_rart_epoch10_checkpoint.pth"

torch.save(
    {
        "model_state_dict": partition_rart.state_dict(),
        "epoch": 10,
        "test_accuracy": 44.62,
        "train_accuracy": 53.62,
        "params": 11272176,
        "config": {
            "image_size": 224,
            "patch_size": 16,
            "num_classes": 100,
            "embed_dim": 384,
            "anchor_dim": 128,
            "depth": 6,
            "num_heads": 6,
            "anchor_heads": 4,
            "num_anchors": 16,
            "mlp_ratio": 3.0
        }
    },
    checkpoint_path
)

print(f"Saved: {checkpoint_path}")

Saved: /kaggle/working/partition_rart_epoch10_checkpoint.pth


In [114]:
research_summary = r"""
# Partitioned Local-Global-Local Vision Transformer

## Research Objective

Investigate whether global communication in Vision Transformers can be performed through a small number of spatially specialized anchor tokens instead of dense patch-to-patch attention.

The initial motivation was that standard ViT global attention has O(N^2) complexity with respect to the number of image patches. The proposed architecture attempts to separate local representation learning from global communication.

The core architecture is:

Patch Embedding
    ↓
Local Patch Attention
    ↓
Patch → Regional Anchor
    ↓
Global Anchor → Anchor Attention
    ↓
Regional Anchor → Patch
    ↓
FFN
    ↓
Next Block

## Baseline

Dataset:
- CIFAR-100
- 50,000 training images
- 10,000 test images
- 100 classes
- Images resized to 224×224
- Patch size: 16×16
- 196 patches

ViT configuration:
- Embed dimension: 384
- Depth: 6
- Heads: 6
- MLP ratio: 4

Baseline results:
- Parameters: 11,057,380
- Inference: 201.345 ms
- Peak GPU memory: 2039.25 MB
- Throughput: 635.73 images/sec
- Test accuracy after 10 epochs: 45.44%

## Initial Anchor-Based RART

The first architecture used local patch attention together with a compressed anchor pathway.

Configuration:
- 16 anchors
- Anchor dimension: 128
- Patch dimension: 384
- Depth: 6

Results:
- Parameters: 11,272,176
- Test accuracy: 42.21%
- Inference: 212.667 ms
- Peak memory: 2619.68 MB
- Throughput: 601.88 images/sec

A 32-anchor version was also tested.

Results:
- Parameters: 11,274,224
- Test accuracy: 39.73%
- Inference: 215.096 ms
- Peak memory: 2802.46 MB
- Throughput: 595.08 images/sec

## Anchor Collapse Investigation

The lower performance led to an investigation of anchor representations.

For the trained 32-anchor model:

Final anchor cosine similarity:
- Block 1: 0.9887
- Block 2: 0.9865
- Block 3: 0.9900
- Block 4: 0.9920
- Block 5: 0.9935
- Block 6: 0.9950

Patch-to-anchor attention was nearly uniform:
- Mean attention entropy: 5.2720
- Maximum possible entropy for 196 patches: approximately 5.278
- Effective patches attended: 194.82 / 196
- Mean similarity between anchor attention maps: 0.9995

This indicated that different anchors were receiving almost identical global summaries.

The collapse originated at Patch → Anchor routing.

## Spatial Routing Investigation

Raw regional patch features were already diverse:
- Mean regional feature similarity: 0.3466

A hard local 5×5 routing constraint produced:
- Mean anchor attention-map similarity: 0.0389
- Effective patches attended: 16 / 196

This demonstrated that spatial constraints could force anchor specialization.

## Local-Global-Local Architecture

A Local → Global → Local architecture was developed:

1. Local patch attention
2. Local Patch → Anchor routing
3. Global Anchor → Anchor self-attention
4. Local Anchor → Patch routing
5. Patch FFN

The 5×5 local-routing version achieved:

- Parameters: 11,272,176
- Test accuracy: 45.38%
- Inference: 399.040 ms
- Peak memory: 4000.50 MB
- Throughput: 320.77 images/sec

Anchor similarity after training:
- Block 1: 0.2619
- Block 2: 0.2550
- Block 3: 0.2976
- Block 4: 0.3295
- Block 5: 0.3682
- Block 6: 0.4227

This showed substantially greater anchor specialization than the unrestricted routing model.

## Partition Routing

The next architecture replaced overlapping local routing with a fixed non-overlapping spatial partition.

For a 14×14 patch grid:
- 196 total patches
- 16 spatial regions
- Each patch belongs to exactly one region
- Region sizes range from 9 to 16 patches
- Mean region size: 12.25 patches

The partitioned architecture uses:

Local Patch Attention
    ↓
Patch → Assigned Regional Anchor
    ↓
Global Anchor Self-Attention
    ↓
Assigned Anchor → Patch
    ↓
FFN

The partition is cached and shared across all blocks.

## Partition RART Results

Configuration:
- Image size: 224×224
- Patch size: 16
- 196 patches
- Embed dimension: 384
- Anchor dimension: 128
- Depth: 6
- Patch heads: 6
- Anchor heads: 4
- Anchors: 16
- MLP ratio: 3

Parameters:
- 11,272,176

Inference:
- 221.430 ms
- 3982.33 MB peak GPU memory
- 578.06 images/sec

Training:
- Epoch 1: 16.60% test accuracy
- Epoch 2: 24.19%
- Epoch 3: 28.90%
- Epoch 4: 32.55%
- Epoch 5: 35.73%
- Epoch 6: 38.35%
- Epoch 7: 40.08%
- Epoch 8: 42.19%
- Epoch 9: 42.85%
- Epoch 10: 44.62%

Final:
- Train accuracy: 53.62%
- Test accuracy: 44.62%

## Current Comparison

| Model | Parameters | Test Accuracy | Inference |
|---|---:|---:|---:|
| ViT | 11.06M | 45.44% | 201.3 ms |
| MeanPool ViT | 11.06M | 44.59% | — |
| RART-16 | 11.27M | 42.21% | 212.7 ms |
| RART-32 | 11.27M | 39.73% | 215.1 ms |
| Local-Global-Local | 11.27M | 45.38% | 399.0 ms |
| Partition RART | 11.27M | 44.62% | 221.4 ms |

## Current Interpretation

The unrestricted anchor routing mechanism caused different anchors to converge toward highly similar representations. This corresponded with lower classification performance.

Introducing explicit spatial routing substantially improved anchor diversity and recovered near-baseline classification accuracy.

The partitioned model reaches 44.62% test accuracy compared with 45.44% for the baseline ViT, a difference of 0.82 percentage points.

The current results therefore support the hypothesis that spatially constrained anchor routing can preserve useful regional specialization while still providing global communication through anchor self-attention.

## Computational Observation

At 224×224:

ViT:
- 196 patches
- Dense global attention contains approximately 196² = 38,416 patch-patch pair interactions per attention layer.

Partition routing:
- 196 patches
- 16 anchors
- Approximately 12.25 patches per region
- Patch-to-anchor communication is restricted to the assigned region.
- Global communication occurs between the 16 anchors.

The current implementation is not yet faster than ViT:
- ViT: 201.3 ms
- Partition RART: 221.4 ms

Therefore, computational superiority has not been demonstrated at 224×224.

## Important Literature Consideration

Anchor-based token bottlenecks are not themselves novel. AnchorFormer and related architectures already investigate anchor tokens as a mechanism for reducing the cost of global ViT attention.

Therefore, the current work should not claim novelty simply from:
- using anchors
- replacing quadratic attention with anchors
- global communication through a small latent set

The potentially differentiating component is the explicit spatial partitioning of patch-to-anchor communication combined with the Local → Global → Local architecture and the resulting preservation of anchor specialization.

This requires further literature comparison and direct implementation comparisons before making a novelty claim.

## Next Experiment

The original hypothesis concerns resolution scaling.

The next experiment should compare ViT and Partition RART at increasing image resolutions:

- 224×224
- 384×384
- 512×512
- potentially 768×768

The number of anchors should scale with spatial regions rather than remaining fixed indefinitely.

Metrics:
- Accuracy
- Inference latency
- Throughput
- Peak GPU memory
- Training time
- Number of patches
- Effective global interactions
- Anchor diversity

The key hypothesis is:

At lower resolution, dense ViT attention may remain competitive or superior.

At higher resolution, partitioned local-global-local routing may scale more favorably because global communication is performed through a much smaller anchor graph rather than dense patch-to-patch attention.

This remains a hypothesis and must be experimentally validated.

## Conclusion

The experiments show that naive anchor-based compression can suffer from anchor collapse, where multiple anchors learn nearly identical global summaries. Explicit spatial partitioning changes this behavior by assigning each patch to a single regional anchor, forcing the anchors to encode different spatial regions before communicating globally.

The resulting Partition RART achieves 44.62% test accuracy on CIFAR-100 at 224×224, compared with 45.44% for the baseline ViT, while using a similar parameter budget.

The strongest current evidence is therefore not that the proposed architecture already outperforms ViT, but that spatially constrained anchor routing provides a viable alternative mechanism for local-to-global information exchange without the severe anchor collapse observed in unrestricted routing.

The central unresolved question is whether this advantage becomes more significant as spatial resolution and patch count increase. High-resolution scaling experiments are required to determine whether the proposed architecture offers a genuine computational or representational advantage over dense ViT attention.
"""

summary_path = "/kaggle/working/partition_rart_research_summary.md"

with open(summary_path, "w") as f:
    f.write(research_summary)

print(f"Saved research summary: {summary_path}")

Saved research summary: /kaggle/working/partition_rart_research_summary.md
